# Bitcoin Heist : détection d'adresses associées aux rançongiciels
**Auteurs :** Théo MICHEL & Théo RICO 

---

## 1. Contexte, problématique et hypothèses

Un rançongiciel est un logiciel malveillant qui bloque l'accès à des données ou à des ressources et réclame une rançon. Les paiements en Bitcoin laissent des traces publiques, mais les adresses sont pseudonymes : observer leurs transactions ne permet pas d'identifier directement leurs propriétaires. L'enjeu est donc d'examiner leur activité pour repérer des comportements qui justifient une investigation.

L'article *BitcoinHeist: Topological Data Analysis for Ransomware Detection on the Bitcoin Blockchain* analyse des caractéristiques extraites du graphe Bitcoin et compare notamment Random Forest, XGBoost, DBSCAN et Mapper, des méthodes d'analyse de données. Il distingue la recherche de nouvelles adresses associées à des familles déjà connues et la détection d'une famille encore inconnue. Ces objectifs demandent des expériences différentes : réussir le premier ne démontre pas que le second est résolu. Nous nous appuyons sur cet article pour définir le problème et discuter les limites de notre protocole, sans prétendre reproduire ses expériences.

**Périmètre des données.** Chaque ligne du CSV représente l'activité d'une adresse. Les six caractéristiques numériques décrivent ses flux et certaines propriétés du graphe de transactions. Le fichier ne contient ni les transactions brutes ni le graphe complet. Il comprend 2 916 697 observations, dont 41 413 étiquetées ransomware, correspondant à 20 849 adresses positives distinctes et 28 familles positives. L'article rapporte un jeu combiné de 24 486 adresses réparties entre 27 familles : les effectifs et les unités d'observation ne se confondent donc pas avec ceux de notre dataset.

**Problématique : dans quelle mesure les six caractéristiques d'activité permettent-elles de repérer les observations étiquetées ransomware sur des adresses absentes de l'entraînement, tout en maîtrisant le nombre de fausses alertes ?**

Ce qu'on envisage est de définir des priorités d'investigations. Une alerte désigne une observation à examiner ; elle ne constitue pas une preuve de malveillance. En particulier, `white` signifie « non signalé ransomware dans les sources disponibles ». Ce label ne garantit pas l'innocence d'une adresse et peut contenir des cas non identifiés. Les performances mesurent ainsi la concordance du modèle avec les annotations disponibles.


### 1.1 Hypothèses et questions d'étude

Nous examinons les hypothèses suivantes, sans présumer des résultats.

- **Les classes présentent des distributions différentes**. Même si les caractéristiques individuelles peuvent parfois être différentes entre les deux classes, leurs distributions se recouvrent majoritairement.
- **Une partie du signal présente des intéractions non linéaires**. Combiner plusieurs caractéristiques permet de mieux classer les observations.
- **Certains comportements sont partagés entre adresses**. Notre modèle est en capacité de détection lorsque les adresses évaluées sont absentes de l'entrainement. 
- **Le déséquilibre des classes conduit à un faible rappel**. On pourra jouer avec le seuil de détection afin d'améliorer le compromis entre détection et fausse alerte. 
- **Une représentation différente peut faciliter l'apprentissage**. On va construire deux ratios à partir des features initiales afin d'apporter un gain.

L'étude relève du **Machine Learning**, ou apprentissage automatique, une composante de l'intelligence artificielle (**AI**). Il s'agit d'un **apprentissage supervisé**, car les labels servent à apprendre la distinction entre les deux classes. Une méthode **non supervisée** rechercherait des structures ou des observations atypiques sans utiliser les labels pendant l'apprentissage ; une observation atypique ne serait pas pour autant automatiquement malveillante. Le **Deep Learning (DL)** utilise des réseaux de neurones profonds : cette famille de méthodes n'est pas nécessaire au protocole retenu sur les données tabulaires disponibles.

Nous évaluons la capacité de classement avec l'Average Precision, puis les décisions avec la précision, le rappel et les effectifs d'alertes. La **précision** est la proportion d'alertes portant effectivement un label ransomware ; le **rappel** est la proportion de positifs connus retrouvés.

### 1.2 Organisation et plan d'étude

| Partie | Question traitée |
|---|---|
| 2. Données et intégrité | Que contient le fichier et que peut-on en déduire ? |
| 3. Exploration | Quels déséquilibres, biais de construction et évolutions orientent l'analyse ? |
| 4. Construction des variables | Quelles variables dérivées construire à partir des mesures disponibles ? |
| 5. Apprentissage | Que vaut une première forêt aléatoire sur des adresses non vues ? |
| 6. Seuil et variables | Quel compromis offre le seuil de décision, et quels ratios aident la forêt ? |
| 7. Hyperparamètres | Quels réglages améliorent la forêt, et ce gain est-il stable ? |
| 8. Comparaison avec XGBoost | Le boosting, puis les montants ronds, améliorent-ils la détection ? |
| 9. Importance des variables | Sur quelles caractéristiques les modèles s'appuient-ils ? |
| 10. Généralisation temporelle | Les relations apprises restent-elles utiles sur une période ultérieure ? |
| 11. Conclusion | Qu'avons-nous montré, quelles limites et comment poursuivre ? |

## 2. Données, définitions et contrôles

### 2.1 Environnement et lecture du fichier


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import plotly.express as px
from IPython.display import display
from pathlib import Path
import hashlib
import json
import joblib
import sklearn
import gc
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    precision_score, recall_score, f1_score, average_precision_score,
    precision_recall_curve,
)
import functions as fn

seed = 45

`income` est conservé en **satoshis**, l'unité entière de la source : 100 000 000 satoshis correspondent à 1 BTC. Ce choix facilite l'audit des montants et évite de créer une colonne décimale redondante. La conversion en BTC ne détruit pas systématiquement l'information ; les arrondis d'affichage et la représentation binaire des nombres décimaux demandent simplement de la vigilance.


In [ ]:
dtypes = {
    "address": "string",
    "year": "int16",
    "day": "int16",
    "length": "int32",
    "weight": "float64",
    "count": "int32",
    "looped": "int32",
    "neighbors": "int32",
    "income": "int64",
    "label": "category",
}

data = pd.read_csv("BitcoinHeistData.csv", dtype=dtypes, low_memory=False)
print(f"Shape du dataset : {data.shape[0]:,} lignes, {data.shape[1]} colonnes")
print(f"Mémoire profonde : {data.memory_usage(index=True, deep=True).sum() / 2**20:.2f} MiB")
data.head()

### 2.2 Définition des caractéristiques

Chaque ligne décrit les agrégats d'activité d'une adresse dans une fenêtre quotidienne. Une même adresse peut donc apparaître dans plusieurs lignes à des dates différentes. Les six features ci-dessous sont déjà calculées à partir du graphe de transactions; le CSV ne fournit pas les transactions brutes nécessaires pour reconstruire ce graphe.

| Variable | Description |
|---|---|
| `address` | Identifiant public de l'adresse. À conserver pour regrouper et contrôler les réapparitions, mais à exclure des entrées du modèle. |
| `year`, `day` | Année et numéro du jour dans l'année (1 à 365 ou 366). Ensemble, ils identifient la fenêtre quotidienne. |
| `length` | Longueur du plus long chemin orienté depuis une transaction de départ (*starter transaction*) jusqu'à l'adresse, comptée en transactions non-starter. Une valeur nulle correspond à une adresse directement atteinte depuis une transaction de départ. |
| `weight` | Poids structurel propagé depuis les transactions de départ et agrégé à l'adresse. Ce n'est ni un montant ni une proportion bornée : plusieurs contributions peuvent s'additionner, donc `weight` peut dépasser 1. |
| `count` | Nombre de transactions de départ distinctes pouvant atteindre l'adresse dans le graphe de la fenêtre. Ce n'est pas le nombre total de transactions de l'adresse. |
| `looped` | Nombre de transactions de départ pouvant atteindre l'adresse par plusieurs chemins orientés. La cohérence attendue dans les données est `looped <= count`. |
| `neighbors` | Nombre de transactions qui ont l'adresse parmi leurs sorties, c'est-à-dire le nombre de transactions qui lui versent des fonds dans la fenêtre; ce n'est pas le nombre d'adresses voisines. |
| `income` | Montant total reçu dans la fenêtre, en satoshis. `100 000 000` satoshis = `1 BTC`. Ce montant n'est ni le solde ni le bénéfice de l'adresse. Le montant reste exprimé en satoshis dans cette étude. |
| `label` | Label fourni par les sources de données : `white` ou nom d'une famille de ransomware connue. C'est la cible, jamais une entrée du modèle; `white` signifie non identifié, pas sain confirmé. |

### Conséquences pour l'analyse

- Exclure `address` et `label` des features. Conserver `address` uniquement pour l'audit des observations répétées et les découpages par groupe.
- Utiliser `income` en satoshis dans les graphiques et les modèles. Une conversion ponctuelle en BTC peut servir à commenter un montant, sans devenir une seconde entrée du modèle.
- Les comptages sont non négatifs et entiers; `weight` est décimal et peut être supérieur à 1. Ne pas appliquer de règle arbitraire qui le bornerait à [0, 1].
- Les transformations logarithmiques seront comparées pour l'exploration et les modèles sensibles à l'échelle. Les arbres n'en ont généralement pas besoin; aucune transformation ne justifie à elle seule la suppression d'observations extrêmes.
- `year` et `day` servent d'abord à construire la date et à découper les données dans le temps. Les inclure comme prédicteurs pourrait permettre au modèle d'apprendre des changements historiques propres aux labels.


In [ ]:
raw_numeric_cols = [
    "year", "day", "length", "weight", "count", "looped", "neighbors", "income"
]
integer_cols = [col for col in raw_numeric_cols if col != "weight"]
required_cols = ["address", *raw_numeric_cols, "label"]
features = ["length", "weight", "count", "looped", "neighbors", "income"]

df = data.copy()

### 2.3 Intégrité des données et audit des valeurs

Les contrôles recensent les doublons, les valeurs absentes ou non numériques, les incohérences de date et les relations attendues entre caractéristiques. Ils ne remplacent ni ne suppriment aucune observation.


In [ ]:
dataset_report = fn.check_dataset(df, reference_date="2026-10-01")
label_report = fn.check_label_distribution(df)

Les contrôles exécutés ne détectent ni doublon exact, ni valeur manquante, ni incohérence parmi les règles vérifiées. Aucune ligne n'est supprimée et aucune valeur n'est remplacée. 

## 3. Exploration des observations

### 3.1 Déséquilibre des labels

Nous auditons les valeurs et la distribution des labels avant de choisir une tâche de classification. 


In [ ]:
df["is_ransomware"] = df["label"].ne("white").astype("int8")
counts = df["is_ransomware"].value_counts().reindex([0, 1], fill_value=0)
proportions = df["is_ransomware"].value_counts(normalize=True).reindex([0, 1], fill_value=0) * 100

print("\nDistribution des labels binarisés :")
print(f"- white (non signalé) : {counts[0]:,} lignes ({proportions[0]:.2f}%)")
print(f"- ransomware signalé : {counts[1]:,} lignes ({proportions[1]:.2f}%)")

plt.figure(figsize=(7, 4.5))
ax = sns.barplot(
    x=["white (non signalé)", "ransomware signalé"],
    y=counts.values,
    hue=["white (non signalé)", "ransomware signalé"],
    palette=["#2A6F97", "#D1495B"],
    legend=False,
)
ax.set_yscale("log")
ax.set_ylim(top=counts.max() * 5)
ax.set_title("Distribution des labels (échelle logarithmique)", pad=18)
ax.set_ylabel("Nombre d'observations")
for i, value in enumerate(counts.values):
    ax.text(i, value * 1.15, f"{value:,}\n({proportions.iloc[i]:.2f}%)", ha="center")
plt.tight_layout()
plt.show()


Le fichier contient **2 875 284 observations `white` et 41 413 observations étiquetées ransomware**, soit respectivement **98,58 % et 1,42 %** des lignes. L'échelle logarithmique permet de représenter les deux effectifs sur une même figure : elle facilite la lecture, mais atténue visuellement l'écart entre les classes. Ces nombres décrivent des observations quotidiennes d'adresses, et non un nombre de personnes, de victimes ou de paiements de rançon.

Ce déséquilibre explique pourquoi l'accuracy ne suffit pas pour évaluer le modèle. Une règle qui classe toutes les lignes en `white` atteint déjà **98,58 % d'exactitude**, avec un rappel nul sur les observations positives. Pour l'usage envisagé, nous cherchons surtout à retrouver des observations signalées comme ransomware tout en limitant les vérifications inutiles.

Le regroupement en une cible binaire masque aussi un déséquilibre entre les labels ransomware. D'après le tableau précédent, `paduaCryptoWall`, `montrealCryptoLocker`, `princetonCerber` et `princetonLocky` totalisent **37 553 observations**, soit **90,68 % des positifs**. Plusieurs autres labels ne comptent que quelques lignes. Une bonne performance globale sur la classe positive ne garantit donc pas une détection satisfaisante pour chacun des labels les plus rares. Enfin, `white` désigne une observation non identifiée comme ransomware dans les sources : cette étiquette ne constitue pas une confirmation d'innocence.

### 3.2 Associations entre les caractéristiques

Nous examinons les associations deux à deux entre les six caractéristiques de graphe et la cible binaire. Pearson décrit une association linéaire; Spearman décrit une association monotone fondée sur les rangs. Ces matrices sont descriptives : une corrélation faible avec la cible ne suffit pas à conclure qu'une variable est inutile, ni qu'un modèle linéaire ne peut pas fonctionner.


In [ ]:
corr_pearson = df[features + ["is_ransomware"]].corr(method="pearson")
corr_spearman = df[features + ["is_ransomware"]].corr(method="spearman")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, matrix, title in zip(
    axes,
    [corr_pearson, corr_spearman],
    ["Pearson", "Spearman"],
):
    sns.heatmap(
        matrix,
        annot=True,
        cmap="coolwarm",
        fmt=".2f",
        vmin=-1,
        vmax=1,
        linewidths=0.5,
        ax=ax,
    )
    ax.set_title(title)

fig.suptitle("Associations deux à deux : features et cible binaire")
plt.tight_layout()
plt.show()


Pearson mesure une association linéaire entre deux variables ; Spearman mesure une association entre leurs rangs et permet de repérer des tendances monotones. Les deux cartes apportent donc des informations complémentaires. Les valeurs affichées sont arrondies à deux décimales : un coefficient noté `0,00` ne signifie pas nécessairement une absence exacte d'association.

La relation entre `length` et `count` apparaît dans les deux cartes : **environ 0,70 selon Pearson et 0,74 selon Spearman**. Les adresses plus éloignées des transactions de départ tendent ainsi à être atteintes par davantage de transactions de départ distinctes, sans que cette association établisse une causalité. `count` et `looped` sont également liés (**0,56 et 0,47**). Cette relation est cohérente avec leur construction : les transactions comptées dans `looped` font partie de celles comptées dans `count`. Les deux mesures restent toutefois différentes, car l'une décrit un volume et l'autre la présence de plusieurs chemins.

L'écart le plus marqué concerne `weight` et `neighbors` : **Pearson vaut environ 0,69, contre −0,01 pour Spearman**. Une association linéaire élevée sur les valeurs brutes ne correspond donc pas ici à une tendance régulière sur l'ensemble des rangs. Ce résultat est compatible avec l'influence de quelques grandes valeurs ou de sous-populations particulières ; les cartes seules ne permettent pas d'en identifier la cause. À l'inverse, `length` et `weight` ont un coefficient de Pearson proche de zéro, mais un coefficient de Spearman d'environ **−0,48** : leur relation ne se résume pas à une droite.

Les coefficients associés à `is_ransomware` restent très faibles dans les deux cartes. Cela ne permet pas de conclure que les caractéristiques sont inutiles pour la classification : une corrélation marginale étudie une variable à la fois et peut masquer des seuils, des interactions ou des sous-groupes. Ce constat est compatible avec l'emploi d'arbres de décision, dont les règles combinent plusieurs variables. Enfin, la corrélation entre certaines caractéristiques impose de lire prudemment les importances de la forêt : deux variables partiellement redondantes peuvent se partager l'importance attribuée par le modèle.

### 3.3 Distributions et valeurs extrêmes

Les caractéristiques ont des échelles et des distributions différentes. Nous comparons ici leurs distributions transformées pour mieux voir la masse centrale et les queues. Cette visualisation ne sert pas à déclarer automatiquement les valeurs extrêmes invalides; `weight` est strictement positif et sera affiché en `log10`, tandis que les variables non négatives utilisent `log1p`.


In [ ]:
# Échantillon équilibré par classe, uniquement pour comparer les formes des distributions.
# Les proportions de cet échantillon ne représentent pas celles du dataset.
samples = []
for _, group in df.groupby("is_ransomware", observed=True):
    samples.append(group.sample(n=min(25_000, len(group)), random_state=42))
sample_df = pd.concat(samples, ignore_index=True)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flatten(), features):
    if col == "weight":
        plot_values = np.log10(sample_df[col])
        scale_name = "log10"
    else:
        plot_values = np.log1p(sample_df[col])
        scale_name = "log1p"

    plot_df = sample_df[["is_ransomware"]].copy()
    plot_df["value"] = plot_values
    sns.boxplot(
        data=plot_df,
        x="is_ransomware",
        y="value",
        hue="is_ransomware",
        palette=["#2A6F97", "#D1495B"],
        legend=False,
        showfliers=False,
        ax=ax,
    )
    ax.set_title(f"{scale_name}({col})")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["white", "ransomware"])
    ax.set_xlabel("Label fourni")
    ax.set_ylabel("Valeur transformée")

fig.suptitle("Distributions par classe (échantillon équilibré pour la visualisation)")
plt.tight_layout()
plt.show()


Chaque boîte décrit la moitié centrale des valeurs ; le trait intérieur indique la médiane. L'échantillon comprend jusqu'à **25 000 observations par classe** afin que la classe minoritaire reste lisible. Les points extrêmes sont masqués dans ces graphiques. 

Les distributions sont fortement asymétriques. Les transformations logarithmiques réduisent l'écart entre les petites et les grandes valeurs et rendent la masse centrale plus facile à comparer. `weight` est représenté par `log10`, et les autres variables par `log1p`. Une valeur négative sur l'axe de `log10(weight)` correspond donc à un poids compris entre 0 et 1, et non à une anomalie ou à un montant négatif. Ces transformations servent ici à la visualisation ; elles ne modifient pas les variables brutes utilisées par la forêt.

Pour `income`, les observations positives présentent une médiane plus basse et une dispersion centrale plus faible que les observations `white`, avec un **recouvrement important** des deux distributions. Cette différence apporte une piste de discrimination, mais ne permet pas de définir un montant caractéristique qui suffirait à identifier un ransomware. Pour `length` et `count`, la dispersion centrale des positifs est également plus limitée sur la figure, sans séparation nette entre les classes. La distribution centrale de `weight` est légèrement décalée vers le haut pour les positifs, mais les boîtes se recouvrent largement. Les deux classes ont des quartiles très proches pour `neighbors`.

La boîte de `looped` est écrasée sur zéro dans les deux classes. Il s'agit d'une concentration des quartiles centraux, et non de la preuve que toutes les valeurs sont nulles. L'audit a relevé **2 504 877 zéros sur 2 916 697 lignes**, soit environ **85,9 %** des observations du fichier. Des valeurs non nulles existent donc bien ; elles sont peu visibles avec cette représentation et le masquage des extrêmes. Dans l'ensemble, ces graphiques montrent des différences de distribution exploitables comme hypothèses, mais aucune variable ne fournit à elle seule une séparation évidente des deux classes.

### 3.4 Relations entre paires de caractéristiques

Le pair-plot présente les distributions individuelles et les relations deux à deux. Au maximum 2 000 observations sont retenues par classe pour conserver une figure lisible. L'échantillon et la transformation `log1p` servent à la visualisation ; l'apprentissage utilisera les variables brutes. La version Plotly est disponible avec `plot_type="plotly"` pour le zoom et le survol.


In [ ]:
pair_grid = fn.pair_plot(
    df, n_points=2000, plot_type="seaborn",
    features=features, use_log=True, random_seed=seed,
)

Ce pair-plot représente **4 000 observations**, soit 2 000 par classe, avec une transformation `log1p` pour rendre les différentes amplitudes lisibles. La proportion de points positifs est donc volontairement plus élevée que dans le fichier. La diagonale montre la distribution de chaque variable ; les autres cases montrent leurs relations deux à deux. `income` reste exprimé en satoshis avant la transformation.

Les points des deux classes se recouvrent dans la plupart des projections, notamment celles qui font intervenir `income`. Certaines zones sont plus denses pour une classe, mais aucune frontière simple ne sépare visiblement l'ensemble des observations. Ce résultat explique l'intérêt d'une méthode qui combine plusieurs règles de décision : un montant ou une mesure de graphe isolée ne suffit pas à caractériser toutes les observations positives.

La relation entre `count` et `looped` est particulièrement structurée. Une première concentration apparaît à `looped = 0` ; d'autres observations se situent près de la diagonale, où les deux comptages sont proches. Le nuage reste cohérent avec la contrainte **`looped <= count`**, contrôlée lors de l'audit. Le ratio `looped / count` permet de résumer cette différence entre absence de chemins multiples et forte proportion de transactions de départ atteignant l'adresse par plusieurs chemins. Cette lecture explique le choix du ratio étudié en partie 4 .

Les bandes visibles sur `length` et `neighbors` proviennent de variables discrètes qui prennent souvent les mêmes valeurs. Elles ne correspondent pas automatiquement à des familles de ransomware ou à des groupes indépendants. Les queues de distribution sont également visibles malgré la transformation logarithmique, avec quelques observations très éloignées de la masse centrale. Leur présence ne suffit pas à les qualifier d'erreurs : la visualisation complète ici les contrôles de cohérence, sans justifier une suppression automatique.

### 3.5 Répartition temporelle et réapparition des adresses

L'évolution des effectifs prépare le choix des fenêtres chronologiques. Les réapparitions expliquent pourquoi les observations d'une même adresse doivent rester groupées dans l'expérience initiale.


In [ ]:
year_counts = pd.crosstab(df["year"], df["is_ransomware"]).reindex(columns=[0, 1], fill_value=0)
year_counts.columns = ["white (non signalé)", "ransomware signalé"]
address_counts = df["address"].value_counts()

fig, ax = plt.subplots(figsize=(9, 4.5))
year_counts.plot.bar(ax=ax, logy=True, color=["#2A6F97", "#D1495B"])
ax.set(xlabel="Année", ylabel="Nombre d'observations (échelle log)",
       title="Disponibilité des observations par période et label")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
display(year_counts)
display(pd.Series({
    "Adresses distinctes": address_counts.size,
    "Adresses observées plusieurs fois": int(address_counts.gt(1).sum()),
    "Observations au-delà de la première par adresse": int(address_counts.sub(1).sum()),
}, name="Effectif").to_frame())


Les effectifs positifs changent fortement selon l'année : **65** observations en 2011, **714** en 2012, **7494** en 2013, **10319** en 2014, **3701** en 2015, **15631** en 2016, **3486** en 2017 et seulement **3** en 2018. L'année 2016 représente à elle seule environ **37,7 % des observations positives** du fichier. Un score calculé sur l'ensemble des années agrège donc des périodes de taille et de composition très différentes.

Les effectifs `white` sont beaucoup plus réguliers : **365 000 observations par an de 2012 à 2017**, puisque le dataset contient 1000 observations `white` par jour. De même, les variations des effectifs positifs dépendent des campagnes présentes et de la couverture des sources. Elles ne permettent pas de conclure directement à une augmentation ou à une diminution du nombre réel de rançons. Les trois positifs disponibles en 2018 rendent notamment cette seule année trop fragile pour évaluer précisément le rappel.

### 3.6 Biais de construction du fichier

La partie 3.5 a montré que les volumes varient selon l'année et que des adresses réapparaissent. Trois propriétés supplémentaires du fichier conditionnent la manière d'évaluer les modèles. Ce ne sont pas des erreurs de saisie : elles découlent de la construction du jeu, qui réunit trois listes d'adresses de rançongiciels (études de Montréal, Princeton et Padoue) et un échantillon d'observations non signalées.

1. **Échantillonnage des `white`** : combien d'observations non signalées par jour, et combien d'observations ransomware ?
2. **Réapparitions positives** : quelle part des observations ransomware provient d'adresses vues plusieurs fois ?
3. **Observations indiscernables** : combien d'observations ransomware ont exactement les mêmes six mesures qu'au moins une observation `white` ?

In [ ]:
# Calculs sur des objets temporaires : df n'est pas modifié.
is_rw = df["is_ransomware"].eq(1)
white_per_day = df.loc[~is_rw].groupby(["year", "day"]).size()
rw_per_day = (df.loc[is_rw].groupby(["year", "day"]).size()
              .reindex(white_per_day.index, fill_value=0))
rw_from_repeated = df.loc[is_rw, "address"].map(address_counts).gt(1)
labels_per_address = df.groupby("address", observed=True)["is_ransomware"].nunique()

# Un vecteur de six mesures est ambigu s'il apparaît avec les deux labels.
patterns = df.groupby(features, observed=True)["is_ransomware"].agg(["size", "sum"])
ambiguous = patterns[(patterns["sum"] > 0) & (patterns["sum"] < patterns["size"])]

display(pd.Series({
    "white par jour (médiane)": white_per_day.median(),
    "jours avec exactement 1 000 white (%)": 100 * white_per_day.eq(1000).mean(),
    "ransomware par jour (médiane)": rw_per_day.median(),
    "ransomware par jour (90e centile)": rw_per_day.quantile(0.9),
    "ransomware par jour (maximum)": rw_per_day.max(),
    "jours sans aucun ransomware (%)": 100 * rw_per_day.eq(0).mean(),
    "lignes ransomware issues d'adresses répétées (%)": 100 * rw_from_repeated.mean(),
    "adresses portant les deux labels": int(labels_per_address.gt(1).sum()),
    "lignes ransomware au vecteur identique à un white (%)":
        100 * ambiguous["sum"].sum() / is_rw.sum(),
}, name="valeur").to_frame().round(2))

# Volume quotidien par classe et part de positifs par année.
daily = pd.DataFrame({"white": white_per_day, "ransomware": rw_per_day})
daily.index = (pd.to_datetime(daily.index.get_level_values("year").astype(str), format="%Y")
               + pd.to_timedelta(daily.index.get_level_values("day") - 1, unit="D"))
share_per_year = 100 * df.groupby("year")["is_ransomware"].mean()

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), gridspec_kw={"width_ratios": [2, 1]})
for col, color in [("white", "#2A6F97"), ("ransomware", "#D1495B")]:
    axes[0].plot(daily.index, daily[col].rolling(30, min_periods=1).mean(),
                 color=color, label=f"{col} (moyenne glissante 30 j)")
axes[0].set_yscale("symlog", linthresh=1)
axes[0].set(title="Observations par jour", ylabel="Nombre d'observations (symlog)")
axes[0].legend()
axes[1].bar(share_per_year.index.astype(str), share_per_year.values, color="#D1495B")
axes[1].axhline(100 * is_rw.mean(), color="gray", linestyle="--", label="Moyenne du fichier")
axes[1].set(title="Part de lignes ransomware par année", ylabel="%")
axes[1].legend()
plt.tight_layout()
plt.show()

* **Les `white` sont un échantillon, pas la population.** Le fichier contient environ 1 000 observations `white` par jour, un nombre quasi constant, alors que toutes les apparitions connues de ransomware sont conservées. La prévalence de 1,42 % est donc un artefact de construction. L'article (section VI) indique environ 200 000 transactions par jour, avec environ 5 adresses chacune : dans le graphe réel, la part d'adresses ransomware est de plusieurs ordres de grandeur plus faible. **Conséquence :** une précision mesurée sur ce fichier n'est pas celle qu'obtiendrait un analyste sur la blockchain complète. Elle sert à comparer des modèles entre eux, et à les comparer à la prévalence du test.
* **La prévalence dépend de la période.** Environ 31 % des jours ne contiennent aucune observation ransomware, alors que le nombre de `white` reste fixe. `year` et `day` encodent donc directement la probabilité d'être positif : un modèle qui les utiliserait apprendrait *quand* les sources ont collecté des adresses, et non *comment* une adresse se comporte. Cela confirme leur exclusion des entrées (partie 4.1).
* **Les réapparitions concernent surtout les positifs.** Plus de la moitié des lignes ransomware proviennent d'adresses présentes dans plusieurs fenêtres, alors qu'aucune adresse ne porte les deux labels. Un découpage aléatoire des lignes permettrait de reconnaître une adresse déjà vue plutôt qu'un comportement, ce qui justifie le découpage par adresse de la partie 5.2.
* **Une partie du problème est indécidable avec ces six mesures.** Environ un tiers des lignes ransomware ont exactement le même vecteur de mesures qu'au moins une ligne `white`. Aucun classifieur ne peut séparer ces observations, ce qui plafonne la précision atteignable à rappel élevé. Ces lignes peuvent aussi correspondre à des paiements de rançon jamais répertoriés, puisque `white` signifie « non signalé ».

### 3.7 Évolution des caractéristiques dans le temps

Le fichier couvre 2011 à 2018. Si le comportement des adresses `white` ou ransomware change au cours du temps, un modèle entraîné sur une période peut se dégrader sur une autre. Nous regardons la médiane annuelle de chaque mesure par classe, puis la succession des familles de rançongiciels.

In [ ]:
# Date de chaque observation, réutilisée en partie 10 ; df n'est pas modifié.
obs_date = (pd.to_datetime(df["year"].astype(str), format="%Y")
            + pd.to_timedelta(df["day"] - 1, unit="D"))

# Médiane annuelle de chaque mesure, par classe. looped vaut 0 en médiane chaque
# année : on trace plutôt la part d'observations avec looped > 0. Les années avec
# moins de 100 observations ransomware (2011, 2018) sont masquées pour cette classe.
yearly = df.groupby(["year", "is_ransomware"], observed=True)[features].median()
yearly["looped"] = 100 * df["looped"].gt(0).groupby(
    [df["year"], df["is_ransomware"]], observed=True).mean()
yearly["income"] = yearly["income"] / 100_000_000  # affichage en BTC
n_yearly = df.groupby(["year", "is_ransomware"], observed=True).size()
yearly = yearly.where(n_yearly.reindex(yearly.index) >= 100)

titles = {"looped": "Part d'observations avec looped > 0 (%)",
          "income": "Médiane annuelle de income (BTC)"}
fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=True)
for ax, col in zip(axes.flatten(), features):
    for cls, color, name in [(0, "#2A6F97", "white"), (1, "#D1495B", "ransomware")]:
        serie = yearly.xs(cls, level="is_ransomware")[col]
        ax.plot(serie.index, serie.values, marker="o", color=color, label=name)
    ax.set_title(titles.get(col, f"Médiane annuelle de {col}"))
    if col == "income":
        ax.set_yscale("log")
axes[0, 0].legend()
fig.suptitle("Dérive temporelle des mesures (médianes par année)")
plt.tight_layout()
plt.show()

# Succession des familles : observations par trimestre pour les 6 familles principales.
top_families = df.loc[is_rw, "label"].value_counts().head(6).index.astype(str)
top_rows = df["label"].isin(top_families)
rw_quarter = (pd.DataFrame({"family": df.loc[top_rows, "label"].astype(str),
                            "quarter": obs_date[top_rows].dt.to_period("Q").dt.to_timestamp()})
              .groupby(["quarter", "family"]).size().unstack(fill_value=0))
ax = rw_quarter.plot.area(figsize=(15, 4.5), alpha=0.85, linewidth=0, colormap="tab10")
ax.set(title="Observations ransomware par trimestre — 6 familles principales",
       xlabel="Trimestre", ylabel="Nombre d'observations")
ax.legend(title="Famille", ncols=3)
plt.tight_layout()
plt.show()

display(pd.DataFrame({"label": df.loc[is_rw, "label"], "address": df.loc[is_rw, "address"],
                      "date": obs_date[is_rw]})
        .groupby("label", observed=True)
        .agg(observations=("address", "size"), adresses=("address", "nunique"),
             premiere=("date", "min"), derniere=("date", "max"))
        .sort_values("observations", ascending=False).head(10))

* **Les `white` dérivent.** Les médianes de `length`, `count` et `weight` des observations non signalées changent nettement selon l'année : `length` médian passe de 2 en 2012 à 44 en 2015, et `count` atteint 6 en 2015. La part d'observations `white` avec `looped > 0` dépasse 28 % en 2014-2015, contre moins de 13 % les autres années. La frontière entre les deux classes n'est donc pas fixe.
* **Les familles se succèdent.** CryptoLocker (2013-2014) est remplacé par CryptoWall (2014-2015), puis Locky et Cerber dominent 2016-2017. L'article (section VI, *Temporal variation*) fait la même observation et entraîne ses modèles sur une fenêtre glissante des derniers jours pour cette raison.
* **Conséquence pour l'évaluation :** le découpage par adresse des parties 5 à 8 mélange toutes les périodes dans l'apprentissage et dans le test. Il mesure la capacité à reconnaître des adresses *de familles et d'époques déjà vues*. La partie 10 évalue le scénario chronologique.

## 4. Construction et sélection des caractéristiques

### 4.1 Variables de référence, unités et normalisation

Une **caractéristique**, ou *feature*, est une quantité fournie au modèle pour produire une prédiction. Les six caractéristiques originales restent la référence : `length`, `weight`, `count`, `looped`, `neighbors` et `income`. L'adresse sert à regrouper les observations ; les dates permettent de les situer dans le temps ; `label` et `is_ransomware` définissent la cible à prédire. Ces identifiants, dates et labels ne sont pas fournis à la forêt.

`income` est conservé en **satoshis entiers (`int64`) dans les données sources**. Un bitcoin vaut 100 000 000 satoshis. Ce choix conserve l'unité du fichier, facilite les contrôles et évite une colonne en BTC qui dupliquerait la même information. Pour limiter la mémoire, les matrices d'apprentissage sont converties en `float32`. Cette conversion arrondit environ 58 % des montants, avec une erreur relative inférieure à 10⁻⁷ : elle est sans effet pratique sur les coupures des arbres. Les indicateurs de la partie 4.4, qui demandent des montants exacts, sont calculés sur `income` en entiers.

La **normalisation** modifie l'échelle des caractéristiques, par exemple en soustrayant leur moyenne et en divisant par leur écart-type. Elle est utile pour des méthodes sensibles aux distances ou aux échelles, comme les SVM. Les arbres recherchent des seuils séparément sur chaque caractéristique, ils n'ont donc pas besoin de cela. Les transformations logarithmiques des parties précédentes servent à rendre les distributions lisibles et ne sont pas appliquées automatiquement aux entrées de la forêt.

### 4.2 Deux caractéristiques dérivées

Le **feature engineering** consiste à construire une représentation à partir des mesures disponibles et de leur signification. Nous retenons deux ratios interprétables, calculables avec les seules informations de la journée étudiée.

| Caractéristique | Définition | Hypothèse |
|---|---|---|
| `loop_ratio` | `looped / count`, si `count > 0` | La part des transactions de départ qui atteignent l'adresse par plusieurs chemins peut être plus informative que leurs seuls nombres. |
| `income_per_neighbor` | `income / neighbors`, si `neighbors > 0` | Le montant moyen reçu par transaction entrante peut distinguer certains comportements indépendamment du montant total. |

Ces ratios ne fournissent pas de nouvelle source d'information : ils réexpriment les caractéristiques existantes. Ils peuvent toutefois simplifier une séparation qui demanderait davantage de coupures avec les variables originales. Un dénominateur nul produit une valeur absente, plutôt qu'une valeur zéro supposée. Les données sources ne sont pas modifiées.

In [ ]:
def make_features(data):
    """Construit deux ratios interprétables dans une copie indépendante.
    Un dénominateur nul donne une valeur absente, jamais un zéro inventé.
    """
    result = data[features].copy()
    result["loop_ratio"] = data["looped"].div(data["count"].where(data["count"].gt(0)))
    result["income_per_neighbor"] = data["income"].div(
        data["neighbors"].where(data["neighbors"].gt(0))
    )
    return result

feature_data = make_features(df)
engineered_features = ["loop_ratio", "income_per_neighbor"]
display(feature_data[engineered_features].describe(percentiles=[0.01, 0.5, 0.99]).T)
display(feature_data[engineered_features].isna().sum().rename("missing_values").to_frame())

**Analyse des deux caractéristiques obtenues.** Les ratios sont définis pour les 2 916 697 observations et ne créent aucune valeur manquante. `loop_ratio` est compris entre 0 et 1, conformément à la relation `looped <= count`. Sa médiane est nulle : au moins la moitié des observations ne présente aucune transaction de départ atteignant l'adresse par plusieurs chemins. 

Le montant moyen par transaction entrante est très asymétrique. Sa médiane est d'environ **112,5 millions de satoshis**, contre une moyenne de **2,35 milliards**, soit environ vingt et une fois plus. Le centile 99 atteint **25 milliards de satoshis**. Ces écarts confirment une queue élevée ; ils ne suffisent pas à considérer les grands montants comme des erreurs à supprimer.

### 4.3 Comment mesurer leur apport ?

Nous conservons une expérience de référence sur les six variables, puis comparons quatre représentations avec les mêmes observations d'apprentissage et de validation :

| Représentation | Entrées du modèle |
|---|---|
| Référence | Les six features originales |
| Premier ajout | Les six features et `loop_ratio` |
| Second ajout | Les six features et `income_per_neighbor` |
| Ajout conjoint | Les six features et les deux ratios |

Cette comparaison est réalisée dans les expériences de la partie 6.
Dans le code, `features` contient les six noms de référence et `engineered_features` contient seulement les deux noms des ratios. La représentation complète est donc **`features + engineered_features`**. Des caractéristiques historiques, comme une activité cumulée antérieure, demanderaient un protocole différent. Le CSV ne garantit pas une observation exhaustive de l'activité passée de chaque adresse. 


### 4.4 Indicateurs de montant rond

Une rançon est souvent demandée sous forme d'un montant fixé par l'attaquant. Nous vérifions si `income` prend plus souvent des valeurs « rondes » chez les observations ransomware : multiples exacts de 0,1 BTC (10 000 000 satoshis) ou de 1 BTC. Le test est exact parce qu'il porte sur `income` en satoshis entiers.

In [ ]:
# Indicateurs calculés dans un tableau séparé : df et feature_data ne sont pas modifiés.
round_data = pd.DataFrame({
    "income_multiple_0.1btc": df["income"].mod(10_000_000).eq(0),
    "income_multiple_1btc": df["income"].mod(100_000_000).eq(0),
}).astype(np.float32)
round_features = list(round_data.columns)

round_share = 100 * round_data.groupby(df["is_ransomware"]).mean()
round_share.index = ["white (non signalé)", "ransomware signalé"]
round_share.columns = ["multiple de 0,1 BTC", "multiple de 1 BTC"]
display(round_share.round(2))

ax = round_share.T.plot.bar(figsize=(8, 4.5), color=["#2A6F97", "#D1495B"], rot=0)
ax.set(title="Part des montants reçus ronds, par classe", ylabel="% des observations")
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f %%")
plt.tight_layout()
plt.show()

# Montants les plus fréquents dans chaque classe, en BTC.
display(pd.concat({
    name: (df.loc[df["is_ransomware"].eq(cls), "income"].div(100_000_000)
           .value_counts(normalize=True).head(10).mul(100)
           .rename_axis("income (BTC)").reset_index(name="% des lignes"))
    for cls, name in [(0, "white"), (1, "ransomware")]
}, axis=1).round(3))
print(f"Montant minimal du fichier : {df['income'].min() / 100_000_000:.2f} BTC")

* **Signal :** une part nettement plus grande des observations ransomware reçoit un montant rond (environ 46 % de multiples de 0,1 BTC, contre environ 13 % chez les `white`). Les montants les plus fréquents côté ransomware sont des valeurs comme 0,5, 1, 1,2 ou 2 BTC, cohérentes avec les motifs du tableau I de l'article.
* **Pourquoi une variable dédiée :** les arbres découpent `income` par seuils. Ils ne peuvent pas représenter facilement la propriété « être un multiple de 0,1 BTC », qui concerne des valeurs isolées réparties sur toute l'échelle.
* **Place dans le protocole :** ces indicateurs ont été construits après la sélection des ratios (partie 6.2) et le réglage de la forêt (partie 7). Pour ne pas modifier ces choix a posteriori, ils sont évalués séparément en partie 8.6, en les ajoutant aux deux modèles finaux.
* **Seuil minimal :** aucun montant n'est inférieur à 0,3 BTC. L'article explique avoir retiré les transferts de moins de 0,3 BTC, « car les rançons sont rarement en dessous ». Les petits paiements sont donc absents par construction, et les conclusions ne s'appliquent qu'à cette plage de montants.

## 5. Apprentissage : première forêt aléatoire

L'objectif est d'estimer, à partir de six caractéristiques d'activité, si une observation quotidienne d'adresse présente le label `white` (non signalée) ou un label de rançongiciel connu. Cette étape relève de l'**apprentissage supervisé** : les caractéristiques constituent les entrées et les labels disponibles servent de cible pendant l'entraînement.

Le label `white` correspond à l'absence de signalement dans les sources, les performances mesurent donc la concordance avec les annotations disponibles.

La forêt aléatoire agrège plusieurs arbres de décision. Chaque arbre est entraîné sur un échantillon tiré avec remise ; à chaque nœud, les coupures sont recherchées parmi un sous-ensemble aléatoire des variables. Ces mécanismes diversifient les arbres et réduisent la variance de l'ensemble. L'implémentation retenue moyenne leurs probabilités de classe, puis sélectionne la classe la plus probable.

Soit $T=\{(x^i,y^i)\}_{i=1}^N$ le jeu d'entraînement et $B$ le nombre d'arbres. Chaque arbre $\varphi^b$ est entraîné sur un échantillon bootstrap $T^b$. L'agrégation par vote majoritaire s'écrit :

$$\varphi_B(x)=\arg\max_{y\in\{0,1\}}\sum_{b=1}^{B}\mathbb{I}\!\left(\varphi^b(x)=y\right).$$

L'indicatrice $\mathbb{I}$ vaut 1 lorsque l'arbre prédit la classe $y$, et 0 sinon. Cette formule décrit le vote des classes. Pour l'implémentation utilisée, l'agrégation porte sur les probabilités estimées par les arbres :

$$\widehat{p}(y\mid x)=\frac{1}{B}\sum_{b=1}^{B}\widehat{p}_b(y\mid x),\qquad
\widehat{y}(x)=\arg\max_{y\in\{0,1\}}\widehat{p}(y\mid x).$$

Ces deux règles d'agrégation peuvent produire des prédictions différentes lorsque les feuilles des arbres ne sont pas pures.


### 5.1 Justification des modèles

Random Forest et XGBoost sont adaptés à une classification supervisée sur variables tabulaires et à des relations non linéaires. Random Forest combine des arbres diversifiés par bootstrap et sélection aléatoire de variables. XGBoost ajoute progressivement des arbres pour réduire une fonction de perte, avec des mécanismes de régularisation. Leur pertinence ne garantit pas qu'ils se séparent correctement sur les labels disponibles.

### 5.2 Définition des partitions et expérience de référence

Une **partition** est une répartition des observations entre plusieurs sous-ensembles ayant des rôles définis. L'entraînement sert à apprendre, la validation sert à choisir les hyperparamètres et le seuil, enfin le test sert à évaluer la procédure retenue sans guider ces choix. Un **hyperparamètre** est un réglage choisi avant l'apprentissage, comme la profondeur maximale des arbres.

L'expérience réserve environ 20 % des adresses au test et entraîne la forêt sur les autres. Toutes les observations d'une même adresse restent du même côté. Le pourcentage de lignes peut différer de 20 %, puisque les adresses n'ont pas toutes le même nombre d'observations. Ce test mesure la généralisation à des adresses non vues, et non la prédiction d'un futur non observé.

Les entrées sont `length`, `weight`, `count`, `looped`, `neighbors` et `income` en satoshis. Les contrôles vérifient que chaque jeu contient les deux classes et qu'aucune adresse n'est partagée.

Les résultats présentés dans toute la partie RandomForest seront effectués avec une valeur de `seed=45` afin de faciliter la reproductibilité.


In [ ]:
seed = 45
test_size = 0.20

features = features.copy()
required = [*features, "address", "label"]

X = data[features].to_numpy(dtype=np.float32, copy=True)
y = data["label"].ne("white").to_numpy(dtype=np.int8) #int8 permet d'alléger la mémoire

groups = pd.factorize(data["address"], sort=True)[0].astype(np.int32)  #on affecte un entier unique à chaque adresse pour alléger la mémoire

split = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
train_indices, test_indices = next(split.split(X, y, groups=groups))
Xtrain, Xtest = X[train_indices], X[test_indices]
ytrain, ytest = y[train_indices], y[test_indices]

# Vérifier les deux classes et l’absence d’adresse commune entre les jeux.
if set(np.unique(ytrain)) != {0, 1} or set(np.unique(ytest)) != {0, 1}:
    raise ValueError("Les deux classes doivent être présentes dans train et test.")
shared_addresses = np.intersect1d(groups[train_indices], groups[test_indices])
if shared_addresses.size:
    raise ValueError("Des adresses sont communes aux jeux d’entraînement et de test.")

split_report = pd.DataFrame({
    "lignes": [len(ytrain), len(ytest)],
    "adresses": [pd.Series(groups[train_indices]).nunique(),
                 pd.Series(groups[test_indices]).nunique()],
    "positifs": [int(ytrain.sum()), int(ytest.sum())],
    "positifs (%)": [100 * ytrain.mean(), 100 * ytest.mean()],
}, index=["Entraînement", "Test"])
display(split_report.round(3))


### 5.3 Apprentissage de la forêt

Le modèle comprend 101 arbres de décision. Les coupures maximisent la réduction d'entropie, c'est-à-dire l'amélioration de la séparation des classes au sein des nœuds. Avec six variables explicatives, le réglage `max_features='sqrt'` conduit à examiner normalement deux variables candidates à chaque coupure ; la recherche peut être étendue si aucune coupure valide n'est trouvée.

Les arbres sont entraînés par bootstrap, sans profondeur maximale imposée. Cette configuration permet des partitions détaillées, mais peut aussi apprendre des particularités du jeu d'entraînement. La comparaison des performances d'entraînement et de test est donc nécessaire. Les classes ne sont pas pondérées dans cette première configuration ; la pondération constitue une variante à évaluer sur validation.


In [ ]:
#on créée un dictionnaire des hyperparamètres pour le modèle de référence, pour la reproductibilité.
reference_params = dict(
    n_estimators=101, criterion="entropy", max_features="sqrt", bootstrap=True,
    class_weight=None, random_state=seed, n_jobs=2,
)

#on vérifie que le modèle sauvegardé correspond bien aux données et aux hyperparamètres utilisés pour l'entraînement.
with Path("BitcoinHeistData.csv").open("rb") as source_file:
    data_fingerprint = hashlib.file_digest(source_file, "sha256").hexdigest()
reference_signature = dict(
    data=data_fingerprint, matrix=hashlib.sha256(np.ascontiguousarray(X).tobytes()).hexdigest(),
    learn=hashlib.sha256(np.ascontiguousarray(train_indices).tobytes()).hexdigest(),
    columns=features, parameters=reference_params, sklearn=sklearn.__version__,
)

#on voit si la forêt de référence existe déjà et si elle peut être réutilisée.
reference_path = Path("rf_reference.joblib") #la forêt déjà entrainée
reference_meta_path = Path("rf_reference.meta.json") #les paramètres de la forêt déjà entrainée
reference_meta = json.loads(reference_meta_path.read_text()) if reference_meta_path.exists() else {}
reuse_reference = False
if reference_path.exists() and reference_meta.get("signature") == reference_signature:
    with reference_path.open("rb") as model_file:
        reuse_reference = hashlib.file_digest(model_file, "sha256").hexdigest() == reference_meta.get("model_sha256")

if reuse_reference:
    rf = joblib.load(reference_path)
    print("Forêt de référence enregistrée réutilisée : données et paramètres inchangés.")
else:
    rf = RandomForestClassifier(**reference_params)
    rf.fit(Xtrain, ytrain)
    joblib.dump(rf, reference_path, compress=3) #on pense à sauvegarder le modèle pour les prochaines fois

    #sécurité pour éviter d'utiliser un modèle qui ne correspond pas aux données et aux hyperparamètres.
    with reference_path.open("rb") as model_file:
        model_digest = hashlib.file_digest(model_file, "sha256").hexdigest()
    reference_meta_path.write_text(json.dumps(dict(signature=reference_signature,
        model_sha256=model_digest, origin="trained_with_signature"), indent=2))

training_score = rf.score(Xtrain, ytrain)
generalization_score = rf.score(Xtest, ytest)
print("Exactitude (entraînement) : %g" % training_score)
print("Exactitude (test) : %g" % generalization_score)
print("Erreur de test : %g" % (1 - generalization_score))
print("Exactitude de la stratégie « toujours white » : %g" % np.mean(ytest == 0))

**Interprétation des résultats.**

L'accuracy atteint **99,56 % à l'entraînement** et **98,46 % au test**, soit un écart d'environ **1,10 point**. Cet écart est compatible avec un ajustement aux particularités de l'entraînement.

L'erreur de test est de **1,54 %**. La stratégie de référence consistant à attribuer systématiquement `white` atteint pourtant **98,6413 % d'exactitude**, soit environ **0,185 point de plus** que la forêt. À coût identique pour toute erreur, le modèle n'améliore donc pas ce critère global. Cette comparaison ne signifie pas que ses alertes sont inutiles : la stratégie de référence ne retrouve aucun positif. L'intérêt opérationnel dépend du rappel obtenu et du coût des fausses alertes, évalués dans la section suivante.


### 5.4 Performance de détection et analyse des erreurs

Les tableaux et graphiques suivants évaluent la détection par classe et le compromis entre rappel et précision. 
Dans les tableaux, **VP** désigne les vrais positifs, **FP** les faux positifs, **FN** les faux négatifs et **VN** les vrais négatifs, toujours relativement aux annotations du fichier.


In [ ]:
ypredict = rf.predict(Xtest)

# Repérer la colonne de la classe positive 1 (ransomware signalé).
positive_column = int(np.flatnonzero(rf.classes_ == 1)[0])
yproba = rf.predict_proba(Xtest)[:, positive_column]
misclass = np.not_equal(ypredict, ytest)

cm = confusion_matrix(ytest, ypredict, labels=[0, 1])
TN, FP, FN, TP = cm.ravel() #true negatives, false positives, false negatives, true positives
metrics = pd.Series({
    "Accuracy": rf.score(Xtest, ytest),
    "Précision (classe 1)": precision_score(ytest, ypredict, zero_division=0),
    "Rappel (classe 1)": recall_score(ytest, ypredict, zero_division=0),
    "F1 (classe 1)": f1_score(ytest, ypredict, zero_division=0),
    "Average Precision": average_precision_score(ytest, yproba),
    "Prévalence positive du test": ytest.mean(),
    "FP / TP": FP / TP if TP else np.nan,
    "TN": TN, "FP": FP, "FN": FN, "TP": TP,
    "Nombre d'erreurs": int(misclass.sum()),
}, name="Résultat")
display(metrics.to_frame().round(4))


report = classification_report(
    ytest, ypredict, labels=[0, 1],
    target_names=["white (non signalé)", "Ransomware signalé"],
    output_dict=True, zero_division=0,
)

report_table = pd.DataFrame({
    name: values for name, values in report.items() if name != "accuracy"
}).T.rename(index={
    "macro avg": "Moyenne non pondérée",
    "weighted avg": "Moyenne pondérée",
}, columns={"precision": "Précision", "recall": "Rappel",
            "f1-score": "F1", "support": "Effectif réel"})
display(report_table.style.format({
    "Précision": "{:.4f}", "Rappel": "{:.4f}", "F1": "{:.4f}",
    "Effectif réel": lambda value: f"{value:,.0f}".replace(",", " "),
}))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay(cm, display_labels=["white", "Ransomware"]).plot(
    ax=axes[0], cmap="Blues", values_format="d", colorbar=False,
)
axes[0].set_title("Matrice de confusion — test")
axes[0].set_xlabel("Label prédit")
axes[0].set_ylabel("Label fourni")
precision, recall, thresholds = precision_recall_curve(ytest, yproba)
axes[1].step(recall, precision, where="post", label="Random Forest")
axes[1].axhline(ytest.mean(), color="gray", linestyle="--", label="Prévalence du test")
axes[1].scatter(metrics["Rappel (classe 1)"], metrics["Précision (classe 1)"],
                color="red", s=35, label="rf.predict")
axes[1].set(xlabel="Rappel", ylabel="Précision", title="Courbe précision-rappel",
            xlim=(0, 1), ylim=(0, 1.02))
axes[1].legend()
plt.tight_layout()
plt.show()

**Interprétation des résultats**


La performance doit être examinée par classe, compte tenu de la forte majorité des observations `white`. Les indicateurs sont calculés relativement aux annotations disponibles : une fausse alerte désigne ici une observation `white` signalée par le modèle.

| Indicateur | Interprétation |
|---|---|
| Précision de la classe ransomware | Proportion des alertes correspondant à un label ransomware. |
| Rappel de la classe ransomware | Proportion des positifs connus retrouvés par le modèle. |
| F1 de la classe ransomware | Moyenne harmonique de sa précision et de son rappel. |
| Average Precision (AP) | Résumé du classement des observations sur la courbe précision-rappel, pour plusieurs seuils. |
| FP/TP | Nombre de fausses alertes par positif retrouvé. |

**Lecture du 2ème tableau :** La ligne **moyenne non pondérée** calcule séparément la moyenne des deux précisions, celle des deux rappels et celle des deux F1. Chaque classe y a le même poids, indépendamment de son effectif. Le F1 moyen n'est pas nécessairement la moyenne harmonique de la précision moyenne et du rappel moyen. La **moyenne pondérée** quant à elle pondère chaque indicateur par les effectifs des classes. Elle est donc dominée par les observations `white`. 

Sur 582 826 observations, dont 7 919 positives, la forêt retrouve **657 positifs** et en manque **7 262**. Elle produit **1 734 fausses alertes** pour **2 391 alertes** : la précision est d'environ **27,48 %**, le rappel de **8,30 %** et le F1 de **0,1274**. Environ **72 %** des alertes concernent donc des observations `white` et **92 %** des positifs connus restent non signalés. Le ratio FP/TP est d'environ **2,64**.

L'Average Precision est de **12,56 %**, pour une prévalence positive de référence d'environ **1,36 %**. Le classement contient un signal, mais ce score ne signifie pas que 12,56 % des alertes sont fiables à tout seuil. La moyenne pondérée est dominée par `white` ; le rappel moyen non pondéré est proche de **54 %**, ce qui révèle le déséquilibre entre les deux classes. La configuration actuelle ne suffit donc pas à retrouver la majorité des positifs connus.

La courbe présente un ensemble de compromis : accroître le rappel augmente significativement la charge d'investigation. La partie 6 mesure ce compromis plus en détail, en faisant varier la valeur de seuil de décision. Le choix d'un objectif opérationnel reste à convenir avec l'utilisateur, en fonction de ses besoins. Le point sur la courbe précision-rappel correspond à la décision par défaut du modèle. Un autre seuil doit être choisi sur un jeu de validation, selon un objectif explicite de rappel et coût.

-----


**Exemples d'erreurs.** Le tableau suivant montre deux types d'erreurs de la forêt de référence sur le test : les cinq fausses alertes au score le plus élevé (label `white`) et cinq positifs manqués dont le score est nul. Ces exemples illustrent les tableaux précédents ; ils ne sont pas représentatifs de toutes les erreurs.

In [ ]:
# Exemples : fausses alertes au score le plus élevé, positifs manqués au score nul.
test_view = df.iloc[test_indices][["address", "year", "day", "label", *features]].copy()
test_view["score_ransomware"] = yproba
false_alerts = test_view[(ytest == 0) & (ypredict == 1)].nlargest(5, "score_ransomware")
missed = test_view[(ytest == 1) & (yproba == 0)].head(5)
display(pd.concat({"Fausses alertes": false_alerts, "Positifs manqués (score nul)": missed}))
print(f"Positifs du test avec un score nul : {100 * np.mean(yproba[ytest == 1] == 0):.1f} %")

Les cinq fausses alertes les plus confiantes reçoivent toutes un montant rond (0,8 ou 1,2 BTC), d'une seule transaction de départ (`count` = 1) et sans chemin multiple (`looped` = 0). Elles ressemblent aux motifs les plus fréquents des rançons (tableau I de l'article) : il peut s'agir de paiements ordinaires du même montant, ou de rançons jamais signalées, puisque `white` signifie « non signalé ». Cette observation motive les indicateurs de montant rond de la partie 4.4. À l'inverse, **28,9 % des positifs du test reçoivent un score nul** : aucun des 101 arbres sans limite de profondeur ne les place dans une feuille contenant un positif, et la forêt ne peut plus les distinguer de la masse des `white`. C'est pourquoi la courbe précision-rappel de la référence s'arrête vers 71 % de rappel.

## 6. Amélioration de Random Forest : seuil et caractéristiques

### 6.1 Du score à la décision

La forêt attribue à chaque observation un score entre 0 et 1. Le **seuil de décision** est la valeur à partir de laquelle on déclenche une alerte. Au lieu d'avoir une réponse binaire, le modèle donnera une probabilité, et l'utilisateur est libre de définir le **seuil** au dessus duquel la proba assez haute pour considérer une alerte. Le seuil modifie le nombre d'alertes, donc la précision, le rappel et le F1. Il ne change pas les scores ni leur *Average Precision*, qui mesure la qualité du classement sur l'ensemble des seuils. 

Dans le code ci-dessous, la décision est appliquée dans `decision_metrics` par **`prediction = (scores >= threshold).astype(np.int8)`**. `threshold_table` compte efficacement les résultats à chaque seuil distinct ; cette fonction ne réentraîne pas la forêt.

Sans coût métier défini pour les erreurs, nous comparons plusieurs seuils fixés à l'avance et celui qui maximise le **F1 sur validation**.

In [ ]:
experiment_dir = Path("rf_experiments")
experiment_dir.mkdir(exist_ok=True)
base_params = reference_params.copy()

with Path("BitcoinHeistData.csv").open("rb") as source_file:
    data_fingerprint = hashlib.file_digest(source_file, "sha256").hexdigest()
#on regarde feature_data et pas juste data car le prochain point fera varier les variables d'entrée
feature_fingerprints = {
    column: dict(dtype=str(feature_data[column].dtype), sha256=fn.array_digest(feature_data[column].to_numpy()))
    for column in feature_data.columns
}


def partition_summary(partitions):
    """Résume les statistiques de chaque partition."""
    return pd.DataFrame({name: dict(rows=len(rows), addresses=len(np.unique(groups[rows])),
                                   positives=int(y[rows].sum()), positive_rate_pct=100*y[rows].mean())
                         for name, rows in partitions.items()}).T


def positive_scores(model, matrix, rows, batch_size=50_000):
    """Calcule les scores par lots pour alléger la mémoire; 
    seule la colonne de la classe 1 est gardée."""
    positions = np.flatnonzero(model.classes_ == 1)
    if len(positions) != 1:
        raise ValueError("La classe positive 1 est absente ou ambiguë.")
    scores = np.empty(len(rows), dtype=np.float64)
    for start in range(0, len(rows), batch_size):
        stop = min(start + batch_size, len(rows))
        scores[start:stop] = model.predict_proba(matrix[rows[start:stop]])[:, int(positions[0])]
    return scores


def sample_addresses(rows, n_addresses, random_state):
    """Tire des adresses au hasard et conserve toutes leurs lignes disponibles."""
    codes = np.unique(groups[rows])
    chosen = np.random.default_rng(random_state).choice(codes, min(n_addresses, len(codes)), replace=False)
    return rows[np.isin(groups[rows], chosen)]


def run_trial(name, columns, params, learn_rows, valid_rows, save_model=False):
    """ON garde un seul modèle dans la mémoire"""
    settings = base_params | params #on veut tous les paramètres, et params va écraser les valeurs de base si elles sont présentes
    
    #La sécurité qui permet de savoir si on n'a pas déjà fait ce calcul
    signature = dict(protocol=2, data=data_fingerprint, sklearn=sklearn.__version__,
                     columns=list(columns), parameters=settings,
                     feature_values={column: feature_fingerprints[column] for column in columns},
                     learn=fn.array_digest(learn_rows), valid=fn.array_digest(valid_rows))
    signature = json.loads(json.dumps(signature, sort_keys=True))
    key = hashlib.sha256(json.dumps(signature, sort_keys=True).encode()).hexdigest()[:20]
    prefix = experiment_dir / key
    model_path = prefix.with_suffix(".joblib")
    result_path, scores_path = prefix.with_suffix(".json"), prefix.with_suffix(".npz")
    if result_path.exists() and scores_path.exists() and (not save_model or model_path.exists()):
        record = json.loads(result_path.read_text())
        if record["signature"] != signature:
            raise ValueError("Le cache ne correspond pas à l'expérience.")
        with np.load(scores_path) as saved:
            scores = saved["scores"]
        print(f"{name} : résultat enregistré réutilisé", flush=True)

        
    else:
        print(f"{name} : apprentissage de {len(learn_rows):,} lignes, {len(columns)} variables...", flush=True)
        matrix = feature_data[columns].to_numpy(dtype=np.float32) #encore une fois, feature data permet d'avoir toutes les features, et rend le code plus versatile pour la suite, pour tester l'efficacité de ces deux nouvelles features
        model = RandomForestClassifier(**settings)

        start = time.perf_counter()
        model.fit(matrix[learn_rows], y[learn_rows])
        fit_seconds = time.perf_counter() - start

        scores = positive_scores(model, matrix, valid_rows)

        record = dict(signature=signature, fit_seconds=fit_seconds,
                      nodes=sum(tree.tree_.node_count for tree in model.estimators_),
                      train_rows=len(learn_rows), validation_rows=len(valid_rows),
                      train_positives=int(y[learn_rows].sum()),
                      **fn.decision_metrics(y[valid_rows], scores, 0.5))
        
        if save_model:
            joblib.dump(model, model_path, compress=3)

        np.savez_compressed(scores_path, scores=scores)
        result_path.write_text(json.dumps(record, indent=2))
        del model, matrix
        
        gc.collect()
        print(f"{name} : AP={record['average_precision']:.4f}, durée={fit_seconds:.1f}s", flush=True)
    row = {k: v for k, v in record.items() if k != "signature"}
    row.update(name=name, columns=list(columns), parameters=settings,
                     feature_values={column: feature_fingerprints[column] for column in columns}, cache_key=key)
    return row, scores, model_path


# Les adresses du test de référence sont exclues de toute sélection.
#pour mener à bien la validation, on va diviser le jeu d'entraînement en deux parties : une pour l'apprentissage et une pour la validation. 
train_split = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=seed+1)
fit_positions, validation_positions = next(train_split.split(Xtrain, ytrain, groups[train_indices]))
fit_indices, validation_indices = train_indices[fit_positions], train_indices[validation_positions]
partitions = dict(fit=fit_indices, validation=validation_indices, reference_test=test_indices)
fn.show_table(partition_summary(partitions))

# Les deux modèles fournis proviennent de l'exécution antérieure du notebook.
if rf.get_params() != RandomForestClassifier(**base_params).get_params():
    raise ValueError("Les réglages de référence et ceux de la partie 6 ne correspondent pas.")
del rf
gc.collect()

#si le modèle de validation existe déjà et qu'il correspond aux données et aux hyperparamètres, on le réutilise.
validation_signature = reference_signature | {"learn": fn.array_digest(fit_indices)}
validation_path = Path("rf_validation.joblib")
validation_meta_path = Path("rf_validation.meta.json")
validation_meta = json.loads(validation_meta_path.read_text()) if validation_meta_path.exists() else {}
reuse_validation = False
if validation_path.exists() and validation_meta.get("signature") == validation_signature:
    with validation_path.open("rb") as model_file:
        reuse_validation = hashlib.file_digest(model_file, "sha256").hexdigest() == validation_meta.get("model_sha256")
if reuse_validation:
    validation_model = joblib.load(validation_path)
    print("Forêt de validation enregistrée réutilisée.", flush=True)

else:
    validation_model = RandomForestClassifier(**base_params).fit(X[fit_indices], y[fit_indices])
    joblib.dump(validation_model, validation_path, compress=3)
    with validation_path.open("rb") as model_file:
        model_digest = hashlib.file_digest(model_file, "sha256").hexdigest()
    validation_meta_path.write_text(json.dumps(dict(signature=validation_signature,
        model_sha256=model_digest, origin="trained_with_signature"), indent=2))
if validation_model.get_params() != RandomForestClassifier(**base_params).get_params():
    raise ValueError("La forêt sauvegardée n'a pas les paramètres attendus.")
if validation_model.estimators_[0].tree_.weighted_n_node_samples[0] != len(fit_indices):
    raise ValueError("Le nombre de lignes d'apprentissage du modèle sauvegardé diffère.")
validation_scores = positive_scores(validation_model, X, validation_indices)
del validation_model
gc.collect()

threshold_results = fn.threshold_table(y[validation_indices], validation_scores)
reference_threshold = fn.best_f1_threshold(y[validation_indices], validation_scores)
threshold_comparison = pd.DataFrame([
    fn.decision_metrics(y[validation_indices], validation_scores, value)
    for value in [0.5, 0.3, 0.2, 0.1, reference_threshold]
], index=["0,50", "0,30", "0,20", "0,10", "F1 maximal sur validation"])
fn.show_table(threshold_comparison.drop(columns=["true_negatives", "average_precision"]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(threshold_results.threshold, threshold_results.precision, label="Précision")
axes[0].plot(threshold_results.threshold, threshold_results.recall, label="Rappel")
axes[0].set(xlabel="Seuil", ylabel="Score", ylim=(0, 1.02), title="Compromis sur validation")
axes[0].legend()
axes[1].plot(threshold_results.threshold, threshold_results.alerts)
axes[1].set(xlabel="Seuil", ylabel="Observations signalées", title="Charge d'investigation")
for ax in axes:
    ax.axvline(reference_threshold, color="gray", linestyle="--", label="Seuil F1")
fig.tight_layout()
plt.show()

**Analyse du seuil sur validation :** Les 468 016 observations de validation contiennent 6 583 positifs, soit 1,41 %. Au seuil 0,5, la forêt retrouve **496 positifs**, en manque **6 087** et produit **1 364 fausses alertes**. La précision est de **26,67 %**, le rappel de **7,53 %** et le F1 de **0,1175**. Ce seuil ne signale donc qu'une petite partie des observations positives. Au contraire, pour un seuil de 0.1, la forêt retrouve **1979 positifs**, en manque **4604** mais produit **13 056** fausses alertes ! La précision est divisiée par deux par rapport au seuil 0.5, le rappel est quant à lui 4 fois supérieur, et donne donc une valeur de F1 meilleure (**0.1831**).

Le tableau et les courbes illustrent bien le compromis. Abaisser le seuil augmente nécessairement le nombre d'observations signalées et ne peut pas diminuer le rappel. La précision peut fluctuer : elle n'est pas nécessairement monotone.

Le F1 maximal sur cette validation est obtenu vers **0,1799** : **1 410 vrais positifs**, **6 440 faux positifs** et **5 173 faux négatifs**. Le rappel atteint **21,42 %**, la précision **17,96 %** et le F1 **0,1954**. Par rapport à 0,50, on retrouve **914 positifs supplémentaires**, au prix de **5 076 fausses alertes supplémentaires**. Le volume total d'alertes est multiplié par environ **4,2**. C'est une amélioration du compromis F1, accompagnée d'une augmentation importante du travail de vérification ; près de **78,6 %** des positifs restent malgré tout non signalés.

Le seuil retenu ici est un scénario technique. Sans précision minimale ni capacité d'investigation définie, il n'existe pas de seuil opérationnel justifié par ces seuls chiffres. Son résultat sur la validation, utilisée pour le choisir, est aussi susceptible d'être optimiste. Enfin, cette forêt a appris sur le `fit` réduit parce qu'on a découpé l'ancien `train` en `fit` et `test` : son AP ne doit pas être comparée directement à 0,1256 sans tenir compte des données d'apprentissage et d'évaluation différentes.

### 6.2 Comparaison contrôlée des deux ratios

Nous comparons les six variables seules, puis avec chaque ratio et avec les deux. La construction des ratios est déterministe et n'apprend aucun paramètre sur la validation.

Pour sélectionner ces variantes, le `fit` est à son tour séparé par adresse en apprentissage et validation **internes**. Nous tirons au hasard 120 000 adresses d'apprentissage et 80 000 adresses de validation, sans sélectionner selon les labels ; toutes leurs lignes disponibles sont conservées. Cette étape utilise un budget réduit et sert au développement uniquement. La validation du seuil et le test de référence restent hors de cette recherche.

Les quatre forêts ont les mêmes paramètres et la même graine. Seule la liste des variables change. La représentation ayant la meilleure AP interne est conservée pour la recherche de paramètres.

In [ ]:
# La recherche utilise deux sous-ensembles du fit ; la validation du seuil reste séparée.
search_split = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=seed+2)
learn_positions, search_positions = next(search_split.split(X[fit_indices], y[fit_indices], groups[fit_indices]))
search_learn = sample_addresses(fit_indices[learn_positions], 120_000, seed+3)
search_valid = sample_addresses(fit_indices[search_positions], 80_000, seed+4)

fn.check_group_split(y, groups, dict(search_learn=search_learn, search_valid=search_valid,
                                 threshold_validation=validation_indices, reference_test=test_indices))
fn.show_table(partition_summary(dict(search_learn=search_learn, search_valid=search_valid)))

#on fait l'analyse pour ces 4 cas
feature_sets = {
    "Six variables": features,
    "+ loop_ratio": features + ["loop_ratio"],
    "+ income_per_neighbor": features + ["income_per_neighbor"],
    "+ deux ratios": features + engineered_features,
}
feature_records = []
for name, columns in feature_sets.items():
    record, _, _ = run_trial(name, columns, {}, search_learn, search_valid) #les paramètres par défaut ont un seuil de 0.5
    feature_records.append(record)
feature_results = pd.DataFrame(feature_records)
best_feature_name = feature_results.sort_values("average_precision", ascending=False).iloc[0]["name"]
selected_columns = feature_sets[best_feature_name] #cette variable nous servira dans la suite
fn.show_table(feature_results[["name", "train_rows", "average_precision", "precision", "recall", "f1", "fit_seconds"]])

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(feature_results.name, feature_results.average_precision, color="#2A6F97")
ax.axvline(y[search_valid].mean(), color="gray", linestyle="--", label="Prévalence")
ax.set(xlabel="Average Precision", title="Apport des ratios — mêmes lignes et mêmes paramètres")
ax.legend()
fig.tight_layout()

Les quatre modèles apprennent sur **133 109 observations**, dont **1 947 positives**, et sont évalués sur les mêmes **88 233 observations**, dont **1 114 positives**. La prévalence de cette validation interne est de **1,26 %**. Pour rappel, la prévalence correspond au cas ou on aurait un modèle qui donnerait aucune alerte, donc ne signalerait uniquement des `white`. Enfin, le seuil utilisé pour chacune de ces analyses est de 0.5.

La référence à six variables obtient une AP de **0,0623**. L'ajout de `loop_ratio` seul donne **0,0611** : cette variable ne semble pas améliorer, voire empire l'AP globale. Cela ne prouve pas non plus qu'il soit inutile dans toutes les configurations. L'ajout de `income_per_neighbor` porte l'AP à **0,0849**, soit un gain relatif de **36 %** par rapport à la référence de cette expérience. Au seuil 0,5, la précision passe de **18,01 % à 26,33 %** et le rappel de **5,03 % à 7,09 %**. Le F1 passe de **0,0786 à 0,1117**. Le montant moyen par transaction entrante facilite donc ici le classement, bien qu'il soit entièrement calculé à partir de deux variables déjà disponibles. 
Les huit variables donnent une AP de **0,0844**, très proche de l'ajout du seul `income_per_neighbor`. L'écart de 0,0005 ne permet pas d'affirmer une supériorité générale, puisque l'étude porte sur un échantillon très réduit de tout notre dataset. Nous retenons **les six variables et `income_per_neighbor`**, meilleure représentation sur cette validation interne et plus simple que l'ajout conjoint. `selected_columns` contient les sept entrées effectivement retenues.

Les apprentissages durent une vingtaine de seconde. Ces résultats proviennent d'un échantillon et d'une seule séparation, avec peu de positifs par rapport au fichier complet. Le niveau d'AP ne se compare pas directement à celui des modèles entraînés sur plusieurs millions de lignes. La confirmation à budget complet et les répétitions de la partie 7 permettent d'examiner la portée du gain.

## 7. Réglage des hyperparamètres et stabilité

Un **hyperparamètre** est un réglage fixé avant l'apprentissage, par exemple le nombre d'arbres ou leur profondeur maximale. Le **surapprentissage** désigne l'apprentissage de particularités de l'entraînement qui se transfèrent mal à d'autres observations. Nous allons, dans cette partie, essayer de trouver les hyperparamètres qui conduisent à la meilleure AP.

### 7.1 Recherche ciblée sur un budget limité

Les candidats utilisent la représentation retenue en 6.2 et exactement les mêmes données de recherche. Le critère principal est l'AP de validation interne, adaptée à la classe rare.

| Réglage testé | Rôle et hypothèse |
|---|---|
| Référence | Entropie, feuilles d'au moins 1 observation, profondeur libre |
| `min_samples_leaf=5` | Éviter des décisions fondées sur une observation isolée |
| `max_depth=20`, feuilles de 5 | Limiter davantage la complexité et le coût |
| `class_weight={0: 1, 1: 5}`, feuilles de 5 | Donner cinq fois plus de poids aux observations positives |
| `class_weight="balanced"`, feuilles de 5 | Pondérer les classes selon l'inverse de leurs effectifs d'entraînement |
| `max_features=4`, feuilles de 5 | Donner davantage de variables candidates à chaque coupure |

Nous conservons **101 arbres**, le bootstrap et **`n_jobs=2`** pour chaque essai. `n_estimators` contrôle le nombre d'arbres et le coût ; `max_samples` la taille des tirages bootstrap ; `criterion` le critère de coupure. Leur optimisation, de même qu'une recherche bayésienne, constituerait un élargissement distinct de ce budget.

Les essais utilisent d'abord un échantillon d'adresses pour maîtriser la mémoire et le temps. Le candidat est ensuite réentraîné sur **tout le fit**, et comparé à la référence entraînée sur ces mêmes lignes. Les durées affichées mesurent l'apprentissage seul sur cette machine, hors chargement, prédiction et écriture des fichiers.

In [ ]:
candidate_params = {
    "Référence": {},
    "Feuilles de 5": {"min_samples_leaf": 5},
    "Profondeur 20, feuilles 5": {"max_depth": 20, "min_samples_leaf": 5},
    "Poids positifs x5, feuilles 5": {"class_weight": {0: 1, 1: 5}, "min_samples_leaf": 5},
    "Classes équilibrées, feuilles 5": {"class_weight": "balanced", "min_samples_leaf": 5},
    "4 variables par coupure, feuilles 5": {"max_features": 4, "min_samples_leaf": 5},
}

tuning_records = []
for name, params in candidate_params.items():
    record, _, _ = run_trial(name, selected_columns, params, search_learn, search_valid) # on voit ici que selected_columns va nous peremttre d'utiliser les features de base + income per neighbor
    tuning_records.append(record)
tuning_results = pd.DataFrame(tuning_records)
best_candidate_name = tuning_results.sort_values("average_precision", ascending=False).iloc[0]["name"]
selected_params = candidate_params[best_candidate_name]
fn.show_table(tuning_results[["name", "average_precision", "precision", "recall", "f1", "alerts", "nodes", "fit_seconds"]])

fig, ax = plt.subplots(figsize=(10, 4))
ranked = tuning_results.sort_values("average_precision")
ax.barh(ranked.name, ranked.average_precision, color="#2A6F97")
ax.set(xlabel="Average Precision", title="Recherche ciblée — validation interne au fit")
fig.tight_layout()
plt.show()


Les six configurations utilisent les mêmes **133 109 observations d’apprentissage**, les mêmes **88 233 observations de validation interne** et 101 arbres. La ligne « Référence » de ce tableau désigne donc cette représentation à sept variables mais avec les réglages initiaux. Encore une fois, le seuil utilisé est celui de 0.5.

L’Average Precision passe de **0,0849** pour cette référence à **0,1325** lorsque chaque feuille contient au moins cinq observations. Limiter en plus la profondeur à 20 donne la meilleure AP de l’étude : **0,1339**. Le nombre total de nœuds diminue alors de **536 501 à 237 371**, soit environ **56 %** de moins. Cette forêt plus contrainte apprend donc un classement plus utile sur cette validation, tout en étant plus compacte. Cela est cohérent avec une réduction du surapprentissage de règles très locales. Cette même configuration ne produit que **24 alertes**, dont **19 positives** : sa précision atteint **79,17 %**, mais son rappel n’est que de **1,71 %**. Un meilleur classement ne garantit donc pas une détection satisfaisante avec le seuil par défaut. C’est pour cette raison que l’AP est le critère principal ici : elle évalue le classement sur tous les seuils. Le seuil adapté au candidat est choisi séparément sur la validation complète dans la partie 7.3. La pondération équilibrée retrouve davantage de positifs au seuil (**38,33 %**), mais génère **6 581 alertes**, dont **6 154 fausses alertes**. Sa précision tombe à **6,49 %** et son AP à **0,1001**. Le poids positif multiplié par cinq atteint une AP de **0,1249**. Ces pondérations modifient la distribution des scores : comparer uniquement leur F1 à 0,5 ne permettrait pas de conclure sur leur intérêt opérationnel.

Enfin, les AP de **0,1325**, **0,1331** et **0,1339** sont proches. Une seule séparation interne ne permet pas d’affirmer que la différence entre ces trois réglages est robuste. Nous retenons donc la configuration `max_depth=20` et `min_samples_leaf=5`, puis vérifions son comportement sur plusieurs séparations d’adresses et sur l’ensemble du `fit` dans les sections suivantes.

### 7.2 Sensibilité à trois séparations par adresse

Nous allons comparer la référence et le nouveau candidat sur 3 splits d'un même ensemble de 240 000 adresses du `fit`. Les deux modèles partagent les mêmes lignes et la même graine. Leurs paramètres et représentations sont fixés avant ces répétitions ; nous ne réglons pas de nouveau le seuil.

Cette analyse fait varier à la fois les adresses d'apprentissage et l'aléa de la forêt. Les validations peuvent se recouper : les trois essais ne sont pas indépendants et l'écart-type n'est pas un intervalle de confiance. Ils renseignent sur la sensibilité du résultat, sans démontrer une stabilité dans le temps.

In [ ]:
# Trois séparations d'adresses ; ni le test de référence ni la validation du seuil ne changent.
base_pool = sample_addresses(fit_indices, 240_000, 123)

stability_records = []
for split_seed in [101, 102, 103]:
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=split_seed)
    learn, valid = next(splitter.split(X[base_pool], y[base_pool], groups[base_pool]))
    learn_rows, valid_rows = base_pool[learn], base_pool[valid]
    fn.check_group_split(y, groups, dict(learn=learn_rows, valid=valid_rows))
    for name, columns, params in [
        ("Référence", features, {}), #la référence ne contient que les 6 features initiales et les paramètres par défaut
        ("Candidat retenu", selected_columns, selected_params),
    ]:
        record, _, _ = run_trial(f"{name} avec seed = {split_seed}", columns,
                                 params | {"random_state": split_seed}, learn_rows, valid_rows)
        record.update(model=name, split_seed=split_seed)
        stability_records.append(record)

stability_results = pd.DataFrame(stability_records)
fn.show_table(stability_results[["model", "split_seed", "train_rows", "validation_rows", "average_precision", "f1", "fit_seconds"]])
stability_summary = stability_results.groupby("model").agg(
    ap_mean=("average_precision", "mean"), ap_std=("average_precision", "std"),
    ap_min=("average_precision", "min"), ap_max=("average_precision", "max"),
    fit_seconds_mean=("fit_seconds", "mean"))
fn.show_table(stability_summary)


fig, ax = plt.subplots(figsize=(8, 4))
for name, group in stability_results.groupby("model", sort=False):
    ax.plot(group.split_seed.astype(str), group.average_precision, marker="o", label=name)
ax.set(xlabel="Graine de séparation et d'apprentissage", ylabel="Average Precision",
       title="Sensibilité à trois séparations par adresse")
ax.legend()
fig.tight_layout()
plt.show()


Le candidat utilise les six variables originales et `income_per_neighbor`, avec une profondeur maximale de 20 et au moins cinq observations par feuille. La référence utilise les six variables et les réglages initiaux. Le seuil utilisé pour cette comparaison est 0.5.

Le candidat obtient une AP supérieure lors des **trois essais** : (**0,1187 contre 0,0716**, **0,1216 contre 0,0695**, puis **0,1415 contre 0,0820**). L'AP moyenne passe de **0,0743 à 0,1273**, soit un écart d'environ **0,0529**. L'amélioration observée lors de la recherche se retrouve donc sur ces trois séparations, avec un budget d'apprentissage identique pour les deux modèles. L'écart-type de l'AP est de **0,0067** pour la référence et de **0,0124** pour le candidat. Ce dernier varie davantage en valeur absolue, malgré un meilleur niveau dans chaque essai. Les performances dépendent donc encore des adresses disponibles et de l'aléa d'apprentissage. Les durées d'apprentissage sont identiques pour le candidat et cette référence. Au seuil 0,5, le F1 du candidat reste pourtant inférieur à celui de la référence dans chaque essai : il varie de **0,0274 à 0,0420**, contre **0,0578 à 0,0944**. Ce résultat ne contredit pas le gain d'AP : la régularisation change la distribution des scores et le seuil 0,5 produit ici très peu d'alertes. La qualité du classement et la qualité d'une décision à un seuil donné répondent à deux questions différentes. Le seuil du candidat est déterminé séparément sur la validation complète dands la partie suivante. 

### 7.3 Confirmation et bilan

Dans cette ultime partie, on entraine le candidat sur les 1 865 855 lignes du `fit`. La référence sauvegardée pour le seuil utilise les mêmes lignes et 101 arbres : leur comparaison est donc contrôlée. L'AP sur la validation de 468 016 lignes départage les deux configurations. Pour chacune, le seuil F1 est calculé sur cette validation.

Les scores, paramètres, temps, graines, versions et empreintes des données/partitions sont enregistrés dans `rf_experiments`. Le cache est vérifié pour chaque nouvel essai ; une modification des données, des paramètres, des variables ou des indices entraîne un nouveau calcul. Les forêts sont traitées successivement et les scores sont calculés par lots.

Les deux forêts de référence avaient déjà été entraînées dans le notebook. Leurs scores sont recalculés ici ; les nouveaux essais sont exécutés et enregistrés dans cette étude. Les détails de réutilisation figurent dans l’annexe technique.

In [ ]:
# Confirmation avec la totalité du fit, pour comparer des budgets identiques.
confirmation_record, candidate_validation_scores, candidate_model_path = run_trial(
    "Candidat sur tout le fit", selected_columns, selected_params,
    fit_indices, validation_indices, save_model=True)

candidate_threshold = fn.best_f1_threshold(y[validation_indices], candidate_validation_scores)
validation_comparison = pd.DataFrame([
    dict(model=name, decision=decision, **fn.decision_metrics(y[validation_indices], scores, threshold))
    for name, scores, tuned_threshold in [
        ("Référence", validation_scores, reference_threshold),
        ("Candidat retenu", candidate_validation_scores, candidate_threshold),
    ]
    for decision, threshold in [("Seuil 0,5", 0.5), ("Seuil F1 validation", tuned_threshold)]
])
fn.show_table(validation_comparison.drop(columns="true_negatives"))

# La décision est figée avant tout nouveau calcul sur le test exploratoire.
reference_ap = average_precision_score(y[validation_indices], validation_scores)
candidate_ap = average_precision_score(y[validation_indices], candidate_validation_scores)
final_model_name = "Candidat retenu" if candidate_ap > reference_ap else "Référence"
print("Configuration retenue sur validation :", final_model_name)
print("Variables du candidat :", selected_columns)
print("Paramètres modifiés du candidat :", selected_params)


test_records, test_score_arrays = [], {}
for name, path, columns, threshold in [
    ("Référence", Path("rf_validation.joblib"), features, reference_threshold),
    ("Candidat retenu", candidate_model_path, selected_columns, candidate_threshold),
]:
    model = joblib.load(path)
    matrix = feature_data[columns].to_numpy(dtype=np.float32)
    scores = positive_scores(model, matrix, test_indices)
    test_score_arrays[name] = scores
    for decision, value in [("Seuil 0,5", 0.5), ("Seuil F1 validation", threshold)]:
        test_records.append(dict(model=name, decision=decision,
                                 **fn.decision_metrics(ytest, scores, value)))
    del model, matrix
    gc.collect()
test_comparison = pd.DataFrame(test_records)
fn.show_table(test_comparison.drop(columns="true_negatives"))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name, scores in test_score_arrays.items():
    precision, recall, _ = precision_recall_curve(ytest, scores)
    axes[0].step(recall, precision, where="post", label=name)
    point = test_comparison.query("model == @name and decision == 'Seuil F1 validation'").iloc[0]
    axes[0].scatter(point.recall, point.precision, s=35)
axes[0].axhline(ytest.mean(), color="gray", linestyle="--", label="Prévalence")
axes[0].set(xlabel="Rappel", ylabel="Précision", xlim=(0, 1), ylim=(0, 1.02),
            title="Classement sur le test exploratoire")
axes[0].legend()
final_point = test_comparison.query("model == @final_model_name and decision == 'Seuil F1 validation'").iloc[0]
final_cm = np.array([[final_point.true_negatives, final_point.false_positives],
                     [final_point.false_negatives, final_point.true_positives]], dtype=int)
ConfusionMatrixDisplay(final_cm, display_labels=["white", "Ransomware"]).plot(
    ax=axes[1], cmap="Blues", values_format="d", colorbar=False)
axes[1].set(title=f"{final_model_name} — seuil fixé sur validation", xlabel="Label prédit", ylabel="Label fourni")
fig.tight_layout()
plt.show()

for name, table in dict(thresholds=threshold_comparison, features=feature_results,
                        tuning=tuning_results, stability=stability_results,
                        confirmation_validation=validation_comparison, exploratory_test=test_comparison).items():
    table.to_csv(experiment_dir / f"{name}.csv", index=True)
manifest = dict(data_sha256=data_fingerprint, sklearn=sklearn.__version__, numpy=np.__version__,
                pandas=pd.__version__, split_seed=seed, n_jobs=2,
                feature_selection=best_feature_name, candidate=best_candidate_name,
                columns=selected_columns, parameters=base_params | selected_params,
                reference_threshold=reference_threshold, candidate_threshold=candidate_threshold,
                final_model=final_model_name, candidate_model_path=str(candidate_model_path),
                feature_fingerprints=feature_fingerprints,
                partitions={k: fn.array_digest(v) for k,v in partitions.items()})
(experiment_dir / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
print("Résultats et protocole enregistrés dans", experiment_dir)

# Les importances de la partie 9 restent celles de l'expérience initiale.
rf = joblib.load("rf_reference.joblib")

La comparaison oppose deux forêts entraînées sur les mêmes **1 865 855 observations**, puis évaluées sur la même validation. La référence utilise les six caractéristiques originales ; le candidat ajoute `income_per_neighbor`, limite la profondeur à 20 et impose cinq observations au minimum par feuille. Les deux modèles conservent 101 arbres, `n_jobs=2` et le boostrap.

Sur validation, l'Average Precision passe de **0,1132 à 0,1884**. Le candidat est retenu sur ce critère avant le nouveau calcul sur le test. Chaque seuil est ensuite fixé au maximum du F1 de validation : **0,179868 pour la référence** et **0,133762 pour le candidat**. Ces valeurs traduisent des compromis propres à chaque modèle ; elles ne sont ni des probabilités de malveillance garanties ni des exigences définies avec un utilisateur. Le F1 constitue ici un objectif d'étude, à remplacer par une contrainte de précision ou de charge si un usage opérationnel est fixé.

Sur les **582 826 observations du test**, dont **7 919 positives**, l'AP passe de **0,1194 à 0,2028**. Le gain concerne le classement et ne dépend donc pas seulement du changement de seuil. Avec leurs seuils fixés sur validation, les modèles donnent les résultats suivants : la précision passe de **17,99 % à 30,72 %**, le rappel de **22,04 % à 23,53 %**, et le F1 de **0,1981 à 0,2665**. Le candidat retrouve **1 863 positifs**, contre **1 745** pour la référence, tout en réduisant les fausses alertes de **7 953 à 4 201**. Le nombre total d'alertes diminue de **9 698 à 6 064**. Dans cette expérience, il détecte donc davantage de positifs avec moins d'observations à examiner.

Au seuil de 0,5, le candidat ne retrouve pourtant que **403 positifs**, contre **616** pour la référence : son classement amélioré exige une règle de décision adaptée. Les **657 détections de la partie 5** proviennent d'une autre forêt, entraînée sur davantage de lignes ; elles ne constituent pas le comparateur contrôlé de cette expérience.

L'amélioration complète combine un ratio et une régularisation : elle ne mesure pas séparément leurs effets sur le fit complet. Les limites restent importantes : **76,47 % des positifs ne sont pas retrouvés** et **69,28 % des alertes portent le label `white`**. Enfin, le test a déjà été consulté pendant l'étude initiale : il conserve un statut exploratoire. Ces résultats ne démontrent ni une fiabilité opérationnelle ni une généralisation à des années futures ou à des familles inconnues.

## 8. Comparaison avec XGBoost

**Question : un ensemble d'arbres appris séquentiellement améliore-t-il le compromis entre détection et fausses alertes ?** La comparaison ne suppose pas que l'un des deux modèles soit supérieur.

### 8.1 Protocole de comparaison

XGBoost suit **le même protocole que la forêt**, avec les mêmes lignes à chaque étape :

| Étape | Random Forest (parties 6-7) | XGBoost (cette partie) |
|---|---|---|
| Recherche des réglages | `search_learn` → `search_valid` (7.1) | Mêmes lignes (8.4) |
| Variables | Sélection en 6.2 : six variables + `income_per_neighbor` | Mêmes variables |
| Confirmation | Apprentissage sur tout le `fit`, AP sur la validation (7.3) | Idem, avec arrêt anticipé sur la validation (8.5) |
| Seuil | F1 maximal sur la validation | Idem |
| Évaluation | Test de référence | Idem |

Quatre configurations sont ainsi comparées : forêt de référence, forêt retenue, XGBoost initial (six variables, réglages par défaut) et XGBoost retenu. Les montants ronds de la partie 4.4 font l'objet d'une expérience distincte (8.6).

### 8.2 Hyperparamètres de XGBoost

| Paramètres | Question à traiter |
|---|---|
| `max_depth`, `min_child_weight` | Quel niveau de complexité généralise ? |
| `learning_rate`, `n_estimators` | Quel compromis entre progression et nombre d'arbres ? |
| `subsample`, `colsample_bytree` | Quelle diversification réduit la sensibilité aux observations ? |
| `reg_alpha`, `reg_lambda`, `gamma` | Quelle régularisation limite les coupures peu utiles ? |
| `scale_pos_weight` | Une pondération des positifs aide-t-elle sur validation ? |
| Arrêt anticipé | Quand cesser l'ajout d'arbres selon une validation distincte ? |

Le nombre d'arbres est fixé par **arrêt anticipé** : l'apprentissage s'arrête quand l'AP de validation ne progresse plus pendant 50 itérations. La recherche de la partie 8.4 teste la profondeur, `min_child_weight`, la régularisation L2 et la pondération des positifs. Le pas d'apprentissage (0,1) et le sous-échantillonnage (80 % des lignes et des colonnes) restent fixés ; les explorer élargirait le budget de recherche.

### 8.3 Principe du boosting

La forêt aléatoire (chapitres 11-12) entraîne des arbres profonds **indépendamment** sur des tirages bootstrap, puis moyenne leurs prédictions : l'agrégation réduit surtout la **variance**.

Le boosting (chapitres 9-10) construit au contraire un modèle **additif et séquentiel** d'arbres peu profonds :

$$F_m(x) = F_{m-1}(x) + \eta\, h_m(x),$$

où chaque nouvel arbre $h_m$ est ajusté au gradient négatif de la fonction de perte (ici la *log-loss* binaire), évalué sur les prédictions de $F_{m-1}$. Le pas $\eta$ (*learning rate*) contrôle la contribution de chaque arbre. Chaque arbre corrige les erreurs résiduelles des précédents : le boosting réduit surtout le **biais**. XGBoost ajoute une régularisation explicite de la complexité des arbres ($\gamma$, $\lambda$), le sous-échantillonnage des lignes et des colonnes, et une construction des arbres par histogrammes adaptée à plusieurs millions de lignes.

`GradientBoostingClassifier` (chapitre 9) et XGBoost relèvent de la même famille de méthodes : nous retenons XGBoost, beaucoup plus rapide sur ce volume. L'article utilise XGBoost avec `nrounds = 25` (section VI) dans un protocole par famille et par fenêtre glissante, que nous ne reproduisons pas.

### 8.4 Recherche ciblée sur les mêmes lignes que la forêt

In [ ]:
import xgboost as xgb
from sklearn.metrics import roc_auc_score

xgb_base = dict(
    n_estimators=2000, learning_rate=0.1, max_depth=6, subsample=0.8, colsample_bytree=0.8,
    tree_method="hist", eval_metric="aucpr", early_stopping_rounds=50,
    random_state=seed, n_jobs=-1,
)
xgb_candidates = {
    "Référence XGBoost": {},
    "Profondeur 4": {"max_depth": 4},
    "Profondeur 8": {"max_depth": 8},
    "min_child_weight=10": {"min_child_weight": 10},
    "Régularisation L2 x10": {"reg_lambda": 10},
    "Poids positifs x5": {"scale_pos_weight": 5},
}

# Mêmes lignes de recherche et mêmes variables que la forêt (parties 6.2 et 7.1).
search_matrix = feature_data[selected_columns].to_numpy(dtype=np.float32)
xgb_records = []
for name, params in xgb_candidates.items():
    model = xgb.XGBClassifier(**(xgb_base | params))
    start = time.perf_counter()
    model.fit(search_matrix[search_learn], y[search_learn],
              eval_set=[(search_matrix[search_valid], y[search_valid])], verbose=False)
    fit_seconds = time.perf_counter() - start
    scores = model.predict_proba(search_matrix[search_valid])[:, 1]
    xgb_records.append(dict(name=name, trees=model.best_iteration + 1, fit_seconds=fit_seconds,
                            **fn.decision_metrics(y[search_valid], scores, 0.5)))
del search_matrix, model
gc.collect()

xgb_tuning = pd.DataFrame(xgb_records)
xgb_best_name = xgb_tuning.sort_values("average_precision", ascending=False).iloc[0]["name"]
xgb_selected_params = xgb_candidates[xgb_best_name]
fn.show_table(xgb_tuning[["name", "average_precision", "precision", "recall", "f1", "alerts",
                          "trees", "fit_seconds"]])
print("Configuration XGBoost retenue :", xgb_best_name)
print("Meilleure forêt sur ces mêmes lignes (partie 7.1) : AP =",
      round(tuning_results["average_precision"].max(), 4))

fig, ax = plt.subplots(figsize=(10, 4))
ranked = xgb_tuning.sort_values("average_precision")
ax.barh(ranked.name, ranked.average_precision, color="#D1495B")
ax.axvline(tuning_results["average_precision"].max(), color="#2A6F97", linestyle="--",
           label="Meilleure forêt (7.1)")
ax.set(xlabel="Average Precision", title="Recherche ciblée XGBoost — validation interne au fit")
ax.legend()
fig.tight_layout()
plt.show()

* Sur les mêmes 133 109 lignes d'apprentissage et 88 233 lignes de validation interne que la forêt, les six réglages de XGBoost donnent des AP proches, de 0,096 à 0,104. La configuration `min_child_weight=10` est retenue de justesse (0,1036 contre 0,1024 pour la référence XGBoost) : elle impose un poids minimal dans chaque feuille, ce qui empêche des feuilles construites sur très peu d'observations. Comme pour la forêt (partie 7.1), une seule séparation ne permet pas d'affirmer que cet écart est robuste.
* Sur cet échantillon, **XGBoost reste en dessous de la forêt réglée** (AP 0,134), mais au-dessus de la forêt à réglages initiaux (0,085, partie 7.1).
* La pondération des positifs (`scale_pos_weight=5`) multiplie les alertes au seuil 0,5 (525 contre 16) sans améliorer l'AP : elle déplace les scores sans mieux les classer, comme `class_weight` pour la forêt.
* L'arrêt anticipé retient de 62 à 282 arbres selon leur profondeur : plus les arbres sont profonds, moins il en faut.

### 8.5 Confirmation sur tout le `fit` et comparaison des quatre configurations

XGBoost initial (six variables, réglages de référence) et XGBoost retenu (sept variables, réglage choisi en 8.4) apprennent sur les mêmes **1 865 855 observations** du `fit` que les forêts de la partie 7.3. Pour chaque modèle, le seuil est fixé au F1 maximal de la validation, puis appliqué au test.

In [ ]:
def fit_xgb(table, params):
    # Apprentissage sur le fit, arrêt anticipé sur la validation ; scores de validation et de test.
    matrix = table.to_numpy(dtype=np.float32)
    model = xgb.XGBClassifier(**(xgb_base | params))
    model.fit(matrix[fit_indices], y[fit_indices],
              eval_set=[(matrix[validation_indices], y[validation_indices])], verbose=False)
    return (model, model.predict_proba(matrix[validation_indices])[:, 1],
            model.predict_proba(matrix[test_indices])[:, 1])


def compare_models(model_scores):
    # Seuil au F1 maximal de validation, puis décision sur le test.
    rows = {}
    for name, (valid_scores, test_scores) in model_scores.items():
        threshold = fn.best_f1_threshold(y[validation_indices], valid_scores)
        test = fn.decision_metrics(ytest, test_scores, threshold)
        rows[name] = {
            "AP validation": average_precision_score(y[validation_indices], valid_scores),
            "AP test": test["average_precision"],
            "AUC ROC test": roc_auc_score(ytest, test_scores),
            "Seuil (F1 validation)": threshold,
            "Précision test": test["precision"], "Rappel test": test["recall"],
            "F1 test": test["f1"], "alerts": test["alerts"],
            "true_positives": test["true_positives"], "false_positives": test["false_positives"],
        }
    return pd.DataFrame(rows).T


xgb_initial, xgb_initial_valid, xgb_initial_test = fit_xgb(feature_data[features], {})
xgb_final, xgb_final_valid, xgb_final_test = fit_xgb(feature_data[selected_columns],
                                                     xgb_selected_params)
print(f"Arbres retenus : XGBoost initial {xgb_initial.best_iteration + 1}, "
      f"XGBoost retenu {xgb_final.best_iteration + 1}")

model_scores = {
    "RF référence": (validation_scores, test_score_arrays["Référence"]),
    "RF retenue": (candidate_validation_scores, test_score_arrays["Candidat retenu"]),
    "XGBoost initial": (xgb_initial_valid, xgb_initial_test),
    "XGBoost retenu": (xgb_final_valid, xgb_final_test),
}
model_comparison = compare_models(model_scores)
fn.show_table(model_comparison)

fig, ax = plt.subplots(figsize=(8, 5.5))
colors = ["#9BB8CF", "#2A6F97", "#F2A7B1", "#D1495B"]
for (name, (_, scores)), color in zip(model_scores.items(), colors):
    precision, recall, _ = precision_recall_curve(ytest, scores)
    ax.step(recall, precision, where="post", color=color,
            label=f"{name} (AP = {average_precision_score(ytest, scores):.3f})")
ax.axhline(ytest.mean(), color="gray", linestyle="--", label="Prévalence du test")
ax.set(xlabel="Rappel", ylabel="Précision", xlim=(0, 1), ylim=(0, 1.02),
       title="Courbes précision-rappel — test de référence")
ax.legend()
plt.tight_layout()
plt.show()

* **La forêt retenue reste le meilleur modèle** : AP de test de 0,203, contre 0,159 pour XGBoost retenu, 0,143 pour XGBoost initial et 0,119 pour la forêt de référence. Le classement est le même sur la validation, qui a servi aux choix (0,188 ; 0,147 ; 0,133 ; 0,113).
* **À réglages initiaux, XGBoost bat la forêt** (0,143 contre 0,119) : ses arbres peu profonds et additifs généralisent mieux que des arbres sans limite de profondeur. **Une fois la forêt régularisée** (partie 7), l'ordre s'inverse : la régularisation de la forêt et l'ajout d'`income_per_neighbor` apportent davantage que le passage au boosting.
* **Au seuil de F1 de validation, les trois meilleurs modèles retrouvent presque le même nombre de positifs** (environ 1 865, soit 23,5 % du test). C'est le nombre de fausses alertes qui les distingue : 4 201 pour la forêt retenue, 6 422 pour XGBoost retenu et 7 256 pour XGBoost initial. Pour le même rappel, la forêt retenue lève environ un quart d'alertes en moins que XGBoost retenu.
* La courbe de la forêt de référence s'arrête vers 71 % de rappel, à cause des 28,9 % de positifs au score nul (partie 5.4). La forêt retenue, aux feuilles d'au moins cinq observations, et XGBoost, aux scores additifs, classent toutes les observations.
* L'article rapporte pour Cerber une précision de 0,033 et un rappel de 0,181 avec XGBoost (tableau V). Nos niveaux sont plus élevés, mais la comparaison reste indicative : protocole différent (un modèle par famille, fenêtres glissantes) et prévalence artificielle du fichier (partie 3.6).

### 8.6 Apport des indicateurs de montant rond

Les deux indicateurs de la partie 4.4 sont ajoutés aux sept variables des deux modèles retenus. Chaque modèle est réentraîné avec **exactement ses réglages, ses lignes et sa règle de seuil** : le seul changement est l'ensemble de variables, donc l'écart mesure l'apport des indicateurs.

In [ ]:
round_table = pd.concat([feature_data[selected_columns], round_data], axis=1)

_, xgb_round_valid, xgb_round_test = fit_xgb(round_table, xgb_selected_params)

round_matrix = round_table.to_numpy(dtype=np.float32)
rf_round = RandomForestClassifier(**(base_params | selected_params))
rf_round.fit(round_matrix[fit_indices], y[fit_indices])
rf_round_valid = positive_scores(rf_round, round_matrix, validation_indices)
rf_round_test = positive_scores(rf_round, round_matrix, test_indices)
del rf_round, round_matrix, round_table
gc.collect()

round_comparison = compare_models({
    "RF retenue": model_scores["RF retenue"],
    "RF retenue + montants ronds": (rf_round_valid, rf_round_test),
    "XGBoost retenu": model_scores["XGBoost retenu"],
    "XGBoost retenu + montants ronds": (xgb_round_valid, xgb_round_test),
})
fn.show_table(round_comparison)

* **Les montants ronds améliorent les deux modèles**, sur la validation comme sur le test. Le gain est fort pour XGBoost (AP de test de 0,159 à 0,197 ; validation de 0,147 à 0,176) et plus modeste pour la forêt retenue (test de 0,203 à 0,212 ; validation de 0,188 à 0,190).
* **Le meilleur modèle de l'étude est la forêt retenue avec montants ronds.** Au seuil fixé sur la validation, elle retrouve 1 868 positifs (23,6 %) avec 3 675 fausses alertes, contre 4 201 sans les indicateurs. Sa précision passe de 30,7 % à 33,7 %, soit environ deux fausses alertes par vrai positif.
* **Une explication possible de l'écart entre modèles :** avec des arbres de profondeur 20, la forêt peut déjà isoler une partie des montants fréquents par des coupures successives autour de chaque valeur. XGBoost, avec des arbres de profondeur 6, en est moins capable et profite davantage d'une variable qui rend la propriété explicite.
* Ce gain est mesuré sur un test déjà consulté (statut exploratoire, partie 7.3). Il est cohérent avec la validation, mais intégrer ces indicateurs à la sélection de la partie 6.2, puis reprendre la recherche de la partie 7, serait nécessaire pour le confirmer.

## 9. Importance des variables

### 9.1 Importance des caractéristiques de la forêt de référence

Les importances représentent la part normalisée de réduction d'impureté attribuée à chaque variable lors de l'entraînement. Elles totalisent 1. Une importance de 0,55 signifie qu'environ 55 % de cette réduction est attribuée à la variable ; elle ne signifie ni 55 % d'exactitude, ni 55 % de cas détectés.

In [ ]:
importances = rf.feature_importances_
std = np.std([tree.feature_importances_ for tree in rf.estimators_], axis=0)
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(8, 6))
plt.title("Importance des variables — Random Forest de référence")
plt.bar(range(X.shape[1]), importances[indices], color="r",
        yerr=std[indices], align="center", capsize=4)
plt.xticks(range(X.shape[1]), np.array(features)[indices], rotation=35, ha="right")
plt.xlim([-1, X.shape[1]])
plt.ylabel("Diminution moyenne d'impureté normalisée")
plt.tight_layout()
plt.show()

display(pd.DataFrame({
    "variable": np.array(features)[indices],
    "importance": importances[indices],
    "std_entre_arbres": std[indices],
}).round(5))

**Interprétation de la figure de référence.** Le montant reçu `income` représente **55,52 %** de la réduction d'impureté normalisée, suivi de `weight` (**19,32 %**) et `count` (**10,99 %**). Ces trois caractéristiques concentrent environ **85,82 %** de l'importance. `length` représente **7,75 %**, `looped` **3,34 %** et `neighbors` **3,09 %**. La forêt utilise principalement le montant et certaines propriétés d'agrégation des flux pour partitionner l'échantillon.

Les barres représentent l'écart-type des importances entre arbres. Pour `income`, il est de **0,01495** autour d'une importance de **0,55516** : sa prépondérance est relativement stable à l'intérieur de cette forêt. Cette stabilité ne garantit pas qu'elle se reproduise sur une autre période. Ces chiffres sont issus de la même référence à six variables que les scores de la partie 5. Ils ne décrivent pas les importances du candidat à sept variables retenu en partie 7 ; cette figure conserve son rôle d'explication de l'expérience initiale.

Ces importances n'indiquent ni le sens de l'effet ni une causalité. Les variables à nombreuses valeurs distinctes peuvent être favorisées et les variables corrélées peuvent se partager l'importance. Une faible valeur ne justifie pas automatiquement une suppression. Les performances de la référence restent limitées malgré la stabilité de cette hiérarchie.

### 9.2 Importances des modèles retenus

Nous comparons la forêt de référence (six variables), la forêt retenue et XGBoost retenu (sept variables). Pour XGBoost, l'importance par **gain** mesure l'amélioration moyenne de la perte apportée par les coupures sur une variable. Elle n'est pas comparable en valeur à la réduction d'impureté de la forêt : nous comparons surtout les **classements**.

In [ ]:
candidate_rf = joblib.load(candidate_model_path)
gain = xgb_final.get_booster().get_score(importance_type="gain")
xgb_gain = pd.Series([gain.get(f"f{i}", 0.0) for i in range(len(selected_columns))],
                     index=selected_columns)
importance_table = pd.DataFrame({
    "RF référence (impureté)": pd.Series(rf.feature_importances_, index=features),
    "RF retenue (impureté)": pd.Series(candidate_rf.feature_importances_, index=selected_columns),
    "XGBoost retenu (gain)": xgb_gain / xgb_gain.sum(),
}).sort_values("XGBoost retenu (gain)", ascending=False)
del candidate_rf
gc.collect()
display(importance_table.round(4))

ax = importance_table.plot.barh(figsize=(9, 5), color=["#9BB8CF", "#2A6F97", "#D1495B"])
ax.invert_yaxis()
ax.set(title="Importance des variables", xlabel="Importance normalisée (somme = 1)")
plt.tight_layout()
plt.show()

* **Pour les forêts, le montant domine mais se répartit.** Dans la forêt de référence, `income` concentre 56 % de l'importance. Dans la forêt retenue, cette part se partage presque à égalité entre `income` (0,30) et `income_per_neighbor` (0,29) : le ratio reprend une partie du rôle du montant total, ce qui est cohérent avec son apport en partie 6.2.
* **XGBoost s'appuie davantage sur les variables discrètes** : `neighbors` arrive en tête de son gain (0,25), alors qu'elle est la dernière variable de la forêt de référence (0,03) ; `looped` suit la même tendance. L'importance par impureté favorise les variables continues à nombreuses valeurs distinctes (`income`, `weight`, le ratio), alors que le gain de XGBoost mesure l'amélioration de la perte. Ces écarts rappellent qu'une importance décrit un modèle, pas le phénomène.
* Dans les trois modèles, les deux variables liées au montant (`income`, `income_per_neighbor`) figurent parmi les plus utilisées. Une importance par permutation sur la validation, ou SHAP (chapitre 14), permettrait de mesurer leur effet sur les prédictions (partie 11.3).

## 10. Généralisation temporelle

**Question : les relations apprises restent-elles utiles sur une période ultérieure ?** Les parties 5 à 8 répartissent les adresses sans ordre chronologique, alors que la partie 3.7 a montré une dérive des caractéristiques et une succession des familles. Nous rejouons XGBoost retenu (mêmes variables et mêmes réglages qu'en 8.5), le modèle le plus rapide à entraîner, dans un scénario chronologique :

| Rôle | Période | Usage |
|---|---|---|
| Apprentissage | jusqu'au 31 mars 2016 | Ajustement des arbres |
| Validation | avril à juin 2016 | Arrêt anticipé uniquement |
| Test | juillet 2016 à fin 2018 | Évaluation finale |

La date de coupure est fixée avant de regarder les scores, pour que le test contienne assez de positifs (Cerber, Locky, CryptXXX et DMALockerv3 restent actifs après juillet 2016). Comme dans l'article (section VI-A, étape 7), nous distinguons les adresses déjà rencontrées avant la période de test de celles qui apparaissent pour la première fois.

In [ ]:
# Découpage chronologique : apprentissage, validation puis période ultérieure de test.
dates = obs_date.to_numpy()
cut_validation, cut_test = np.datetime64("2016-04-01"), np.datetime64("2016-07-01")
time_fit = np.flatnonzero(dates < cut_validation)
time_validation = np.flatnonzero((dates >= cut_validation) & (dates < cut_test))
time_test = np.flatnonzero(dates >= cut_test)
fn.show_table(partition_summary({
    "apprentissage (< avril 2016)": time_fit,
    "validation (avril-juin 2016)": time_validation,
    "test (≥ juillet 2016)": time_test,
}))

time_matrix = feature_data[selected_columns].to_numpy(dtype=np.float32)
xgb_time = xgb.XGBClassifier(**(xgb_base | xgb_selected_params))
xgb_time.fit(time_matrix[time_fit], y[time_fit],
             eval_set=[(time_matrix[time_validation], y[time_validation])], verbose=False)
time_scores = xgb_time.predict_proba(time_matrix[time_test])[:, 1]
y_time = y[time_test]
del time_matrix
gc.collect()

# Adresses jamais vues avant la période de test.
new_address = ~np.isin(groups[time_test], groups[np.concatenate([time_fit, time_validation])])


def ranking_metrics(y_true, scores):
    return {"AP": average_precision_score(y_true, scores),
            "AUC ROC": roc_auc_score(y_true, scores),
            "AP du hasard (prévalence)": y_true.mean(),
            "Positifs": int(y_true.sum())}


display(pd.DataFrame({
    "Découpage par adresse (partie 8.5)": ranking_metrics(ytest, xgb_final_test),
    "Chronologique — tout le test": ranking_metrics(y_time, time_scores),
    "Chronologique — adresses nouvelles": ranking_metrics(y_time[new_address],
                                                         time_scores[new_address]),
}).T.round(4))

# Familles positives du test, et présence dans la période d'apprentissage.
labels_all = df["label"].astype(str).to_numpy()
test_families = (pd.Series(labels_all[time_test][y_time == 1]).value_counts()
                 .rename("positifs du test").to_frame())
test_families["présente à l'apprentissage"] = test_families.index.isin(set(labels_all[time_fit]))
display(test_families.head(8))

fig, ax = plt.subplots(figsize=(8, 5.5))
curves = {"Découpage par adresse": (ytest, xgb_final_test, "#2A6F97"),
          "Chronologique (adresses nouvelles)": (y_time[new_address],
                                                 time_scores[new_address], "#D1495B")}
for name, (y_true, scores, color) in curves.items():
    precision, recall, _ = precision_recall_curve(y_true, scores)
    ax.step(recall, precision, where="post", color=color,
            label=f"{name} (AP = {average_precision_score(y_true, scores):.3f})")
    ax.axhline(y_true.mean(), color=color, linestyle=":", linewidth=1)
ax.set(xlabel="Rappel", ylabel="Précision", xlim=(0, 1), ylim=(0, 1.02),
       title="XGBoost retenu : adresses non vues contre période future\n(pointillés : prévalences)")
ax.legend()
plt.tight_layout()
plt.show()

* **La performance s'effondre dans le temps.** Sur la période ultérieure, l'AP de XGBoost retenu tombe à 0,024 pour une prévalence de 0,012 : le modèle fait à peine deux fois mieux que le hasard, contre plus de dix fois avec le découpage par adresse (AP 0,159). L'AUC ROC passe de 0,85 à 0,67. Le résultat est identique sur les seules adresses nouvelles, qui représentent 99 % des positifs du test.
* **Ce n'est pas seulement l'arrivée de familles inconnues.** Cerber et Locky, près de 90 % des positifs du test, sont déjà présentes dans l'apprentissage (premiers mois de 2016). La dégradation vient donc aussi de la dérive des comportements, des `white` comme des familles connues (partie 3.7).
* **La validation est elle-même atypique** : 5,6 % de positifs, contre 1,2 % dans le test. L'arrêt anticipé a donc été réglé sur une période différente de celle évaluée.
* **Conséquence :** le découpage par adresse des parties 5 à 8 surestime nettement la performance sur des paiements futurs. Ce résultat rejoint l'article, qui entraîne ses modèles sur une fenêtre glissante récente (30 à 240 jours), et dont le tableau VII rapporte l'échec de Random Forest et de XGBoost face à de nouvelles familles. Pour une question de coût, seul XGBoost a été évalué dans ce scénario.

## 11. Conclusion

### 11.1 Ce que montrent les résultats

| Expérience | Modèle | AP de test | Lecture |
|---|---|---|---|
| Hasard (prévalence du test) | — | 0,014 | Référence |
| Adresses non vues (partie 7.3) | Forêt de référence, six variables | 0,119 | Signal réel mais modeste |
| Adresses non vues (partie 8.5) | XGBoost retenu | 0,159 | Le boosting bat la forêt non réglée |
| Adresses non vues (partie 7.3) | Forêt retenue | 0,203 | La régularisation et `income_per_neighbor` comptent le plus |
| Adresses non vues (partie 8.6) | Forêt retenue + montants ronds | **0,212** | Meilleur modèle de l'étude |
| Période future (partie 10) | XGBoost retenu | 0,024 (hasard : 0,012) | Le signal se transfère très mal dans le temps |

Les quatre premières lignes portent sur les mêmes 1 865 855 observations d'apprentissage et le même test.

**Réponse à la problématique.** Les six caractéristiques d'activité permettent de repérer une partie des observations ransomware sur des adresses absentes de l'entraînement : le meilleur modèle classe ces observations environ quinze fois mieux que le hasard. Au seuil fixé sur la validation, il retrouve **1 868 des 7 919 positifs du test (23,6 %) avec 3 675 fausses alertes**, soit environ deux fausses alertes par vrai positif. Le gain vient d'abord de la représentation et de la régularisation : `income_per_neighbor`, des feuilles d'au moins cinq observations et une profondeur limitée font passer l'AP de 0,119 à 0,203, puis les montants ronds à 0,212. Le boosting, à budget de réglage comparable, ne fait pas mieux que la forêt régularisée. En revanche, plus des trois quarts des positifs restent non signalés, et **lorsque l'apprentissage précède la période évaluée, le signal devient très faible** : un modèle appris une fois ne suffit pas.

### 11.2 Limites

* **Prévalence artificielle** : environ 1 000 `white` par jour (partie 3.6). Les précisions et charges d'alerte ne sont pas transposables à la blockchain complète.
* **Labels incomplets** : `white` signifie « non signalé », et un tiers des lignes ransomware ont exactement les mêmes mesures qu'une ligne `white`. La performance atteignable avec ces six mesures est plafonnée.
* **Test exploratoire** : le test de référence a été consulté dès la partie 5 ; les écarts observés dessus sont cohérents avec la validation, mais ne constituent pas une évaluation indépendante.
* **Réglage partiel** : la recherche porte sur un échantillon réduit et une seule séparation, avec six configurations par modèle. Les montants ronds n'ont pas été intégrés à la sélection des variables.
* **Un seul découpage chronologique**, avec une seule date de coupure.

### 11.3 Comment poursuivre

1. **Fenêtres glissantes** : réentraîner sur les derniers mois et évaluer sur plusieurs périodes successives, comme l'article. C'est la piste la plus directe après la partie 10.
2. **Intégrer les montants ronds à la sélection** de la partie 6.2, puis reprendre la recherche de la partie 7 sur cette représentation.
3. **Élargir le réglage** : nombre d'arbres, `max_samples`, pas d'apprentissage de XGBoost, avec une recherche aléatoire puis une optimisation bayésienne (chapitre 7), et une dispersion mesurée sur plusieurs séparations.
4. **Expliquer les décisions** : importance par permutation sur la validation, retrait de `income` puis réentraînement, et SHAP (chapitre 14) sur des vrais positifs, fausses alertes et positifs manqués de plusieurs familles.
5. **Diagnostic par population** : rappel par famille, par période et par tranche de montant.
6. **Familles inconnues** : retirer une famille de l'apprentissage et mesurer sa détection, comme le tableau VII de l'article.
7. **Méthodes alternatives** : détection d'anomalies (chapitre 13 : Isolation Forest, LOF, One-Class SVM), en gardant à l'esprit qu'une observation atypique n'est pas forcément un rançongiciel ; DBSCAN et Mapper/TDA comme dans l'article ; un arbre seul ou Naive Bayes comme références simples.

### 11.4 Informations à demander

Il faudrait préciser la provenance et la date d'attribution des labels, la façon dont les observations `white` ont été échantillonnées, la définition exacte des fenêtres d'agrégation et la disponibilité des transactions brutes. Ces transactions permettraient de construire d'autres variables de graphe, comme la distance aux adresses ransomware connues utilisée par l'article (section VII). L'examen de fausses alertes et de positifs manqués vérifiés, ainsi que la capacité réelle d'investigation par jour, permettraient de choisir un seuil fondé sur un besoin opérationnel plutôt que sur le seul F1.

À ce stade, la détection de toutes les activités de rançongiciels sur Bitcoin n'est pas démontrée : nous avons mesuré un signal, ses conditions de validité et ses limites.

## Références méthodologiques

- Akcora, Li, Gel et Kantarcioglu, *BitcoinHeist: Topological Data Analysis for Ransomware Detection on the Bitcoin Blockchain*, fichier [article.pdf](article.pdf), notamment les sections IV à VI et les tableaux V et VII. [Version des auteurs](https://arxiv.org/abs/1906.07852).
- Documentation de [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html), pour l'agrégation, les réglages et les conversions numériques.
- Documentation du [réglage du seuil](https://scikit-learn.org/stable/modules/classification_threshold.html), pour séparer apprentissage des scores et choix d'une décision.
- Documentation du [réglage de XGBoost](https://xgboost.readthedocs.io/en/stable/tutorials/param_tuning.html), pour la complexité, la régularisation et le déséquilibre.


## Annexe A — Traçabilité des calculs

Toutes les forêts ont été réentraînées lors de la dernière exécution complète du notebook. Leurs signatures (empreintes des données, des variables, des partitions, paramètres et version de scikit-learn) sont enregistrées dans `rf_reference.meta.json`, `rf_validation.meta.json` et `rf_experiments/`. Une réutilisation n'a lieu que si toutes ces signatures correspondent ; un cache absent ou incompatible provoque un nouvel entraînement. Les modèles (`.joblib`) et les scores (`.npz`) ne sont pas versionnés : sur une nouvelle machine, le notebook les recalcule. Les essais XGBoost de la partie 8, la forêt avec montants ronds (8.6) et l'expérience chronologique (10) sont recalculés à chaque exécution. Les répétitions de la partie 7 portent sur des échantillons de développement ; le bilan du test reste exploratoire.